# 🧮 08 · Classification I: k-nearest neighbours and naïve Bayes
### From genotypes to cancer risk, from expression profiles to leukaemia type: a lazy classifier, a generative classifier, and why conditional independence turns 3ⁿ parameters into 4n + 1

**BiologyToML** · Part B — Supervised learning · Module 08 of 24

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/08_knn_naive_bayes.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib, scikit-learn (for checking only) |
| **Prerequisites** | [Module 02](02_probability_for_biologists.ipynb) (conditional probability, Bayes' rule) and [Module 05](05_likelihood_mle_map.ipynb) (likelihood, MLE, MAP); Python as in Module 01 |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **02-620 lectures covered** | `003_classification-kNN-NB` slides 1–55 — see [`SLIDE_MAP.md`](../SLIDE_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **Real data:** the Golub et al. (1999) leukaemia expression data (72 patients, original 38/34 train/test
> split; the notebook prints its source, and falls back to a labelled simulated stand-in offline) and the
> six-individual genotype table of the 02-620 slides. **Simulated data (labelled SIMULATED):** the
> three-cell-type marker data used for k-NN, the populations used to illustrate conditional independence,
> the small single-cell marker table of Section 7 and the case/control cohort with correlated SNPs.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** what a classifier is, how training and testing differ, and how the three classifier families of 02-620 (k-NN, naïve Bayes, logistic regression) differ: non-parametric, generative, discriminative.
2. **Implement** k-nearest neighbours from scratch, **interpret** its decision boundaries, and **analyse** how K, feature scaling and the training-set size change accuracy and test-time cost.
3. **Compute** the number of parameters of the full table P(Y | X₁…Xₙ) (3ⁿ), of Bayes' rule without assumptions ((3ⁿ − 1) × 2 + 1) and of naïve Bayes (2·n·2 + 1), and **explain** why conditional independence makes learning possible.
4. **Derive** the maximum-likelihood estimates of naïve Bayes (count ratios) from the log-likelihood and **reproduce** every number of the slides' six-individual genotype example by hand and in code.
5. **Diagnose** the two subtleties of naïve Bayes — zero counts (fixed by MAP pseudo-counts) and violated independence (over-confident probabilities) — and **quantify** them with simulations.
6. **Implement** Gaussian naïve Bayes for continuous features, **apply** it to real leukaemia expression data with genes chosen on the training samples only, and **count** its parameters for k classes and n features.
7. **Connect** naïve Bayes to the course's spine: the two-step template (model, then MLE/MAP) and "conditional independence buys tractability".

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, the Golub data | 5 min |
| 1 · What is classification? | f : X → Y, training vs testing, three kinds of classifier | 8 min |
| 2 · k-nearest neighbours | distance, voting, K, scaling, cost (slides 5–12) | 25 min |
| 3 · Genotypes → cancer | the slides' table; why the full P(Y ∣ X) cannot be learned (slides 14–20) | 12 min |
| 4 · Conditional independence | chain rule, the naïve Bayes assumption, 3ⁿ → 4n + 1 (slides 22–23) | 12 min |
| 5 · The full naïve Bayes model | Bernoulli, multinoulli, indicator notation, the denominator (slides 24–32) | 12 min |
| 6 · MLE by counting | derivation, the worked genotype example, CategoricalNB check (slides 34–46) | 20 min |
| 7 · A second worked example | two binary markers → cell type (slide 47) | 8 min |
| 8 · Subtlety #1: zero counts | MAP with pseudo-counts (slides 48–50) | 12 min |
| 9 · Subtlety #2: dependent features | duplicated features, SNPs in linkage disequilibrium (slide 51) | 12 min |
| 10 · Gaussian naïve Bayes | continuous features on real leukaemia data (slides 52–55) | 20 min |
| Wrap-up | takeaways, checkpoint, discussion, lab note | 4 min |

## 🧫 The biological question

*"Given my genotypes, am I going to develop cancer?"* — that is how the 02-620 lecture opens its naïve-Bayes
story (slide 14). A genome-wide association study (Module 06) finds loci whose genotypes differ between cases and
controls; the natural next question is predictive: from a person's genotypes at a few loci, what is the
*probability* that they are a case? The output is no longer a number on a continuous scale (a phenotype) but a
**category** — healthy or cancer — and predicting categories is called **classification**.

Classification is everywhere in biology (slide 2): disease status from SNPs or gene expression, tumour versus
healthy tissue from pathology images, a diagnosis from medical records, "is this a transcription-factor binding
site?" from sequence features, "which cell type is this?" from single-cell measurements.

This notebook uses one famous real example throughout. In 1999, Golub and colleagues measured the expression of
thousands of genes in bone-marrow and blood samples of patients with two kinds of acute leukaemia — **acute
lymphoblastic leukaemia (ALL)** and **acute myeloid leukaemia (AML)** — and showed that the expression profile alone
can tell the two apart. The distinction matters because the two leukaemias are treated with different chemotherapy
regimens. At the time it relied on an expert combining several laboratory tests. We will classify these real
patients with two of the simplest — and still widely used — classifiers: **k-nearest neighbours** and
**naïve Bayes**.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import itertools
import time
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.naive_bayes import CategoricalNB, GaussianNB

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
We need the slides' six-individual genotype table (`NB_GENOTYPE_TABLE`), the Golub leukaemia loader, z-scoring, the logistic function, a stable log-sum-exp, accuracy, and a function that shades the decision regions of any 2-feature classifier. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_DATA_URL = "https://raw.githubusercontent.com/xulabs/edu/main/ml/tutorial/BiologyToML/datasets/{name}"


CACHE_DIR = os.environ.get("BIO2ML_CACHE", "course_data")


def fetch_course_file(name: str, timeout: float = 60.0) -> str | None:
    """Return a local path to the course data file ``name``, or ``None`` if it cannot be found.

    Looks in the repository's ``datasets/`` folder first (when the notebook runs inside a
    clone of xulabs/edu), then in the download cache, and finally downloads it from GitHub.
    """
    for base in ("../datasets", "datasets", "ml/tutorial/BiologyToML/datasets", CACHE_DIR):
        path = os.path.join(base, name)
        if os.path.exists(path) and os.path.getsize(path) > 0:
            return path
    dest = os.path.join(CACHE_DIR, name)
    os.makedirs(CACHE_DIR, exist_ok=True)
    try:
        with urllib.request.urlopen(REPO_DATA_URL.format(name=name), timeout=timeout) as r, \
                open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return dest
    except Exception as e:  # noqa: BLE001 — any network problem → offline fallback
        print(f"⚠️  Could not obtain {name} ({type(e).__name__}); using a simulated stand-in instead.")
        return None


NB_GENOTYPE_TABLE = np.array([
    [0, 0, 1, 1],
    [1, 0, 2, 1],
    [0, 2, 0, 1],
    [2, 0, 0, 0],
    [2, 1, 2, 0],
    [1, 2, 1, 0],
])


def simulate_golub_like(rng: np.random.Generator) -> dict:
    """Simulated stand-in for the Golub data: 72 samples (47 'ALL', 25 'AML') × 3,571 genes.

    Log10-scale expression with gene-specific means, correlated gene modules and
    60 genes that differ between the classes.  Same keys as ``load_golub``.
    """
    n_all, n_aml, J = 47, 25, 3571
    y = np.r_[np.zeros(n_all, int), np.ones(n_aml, int)]
    base = rng.normal(2.6, 0.45, J)
    modules = rng.normal(0, 1, (len(y), 20))                  # 20 hidden "programmes"
    loading = rng.normal(0, 0.15, (20, J)) * (rng.random((20, J)) < 0.1)
    X = base + modules @ loading + rng.normal(0, 0.25, (len(y), J))
    informative = rng.choice(J, 60, replace=False)
    X[:, informative] += np.outer(y - 0.5, rng.choice([-1, 1], 60) * rng.uniform(0.4, 0.9, 60))
    is_train = np.r_[np.arange(n_all) < 27, np.arange(n_aml) < 11]
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL"), genes=np.array([f"gene{j:04d}" for j in range(J)]),
                is_train=is_train, sample_id=np.arange(1, len(y) + 1), source="SIMULATED stand-in (offline)")


def load_golub() -> dict:
    """Golub et al. (1999) leukaemia expression data, preprocessed as in Dudoit et al. (2002).

    Returns a dict with ``X`` (72 × 3,571, log10 expression), ``y`` (0 = ALL, 1 = AML),
    ``labels``, ``genes`` (Affymetrix probe IDs), ``is_train`` (the original 38/34 split),
    ``sample_id`` and ``source``.  Falls back to ``simulate_golub_like`` when offline.

    Raw intensities were clipped to [100, 16000] before the log10, so the value 2.0 means
    "at or below the detection limit" (about 20 % of all entries).  Some genes are therefore
    constant within a class — per-class variance estimates need a small floor.
    """
    path = fetch_course_file("golub_leukemia.csv.gz")
    if path is None:
        return simulate_golub_like(np.random.default_rng(1999))
    df = pd.read_csv(path)
    return dict(X=df.iloc[:, 3:].to_numpy(float), y=(df["label"] == "AML").to_numpy(int),
                labels=df["label"].to_numpy(), genes=np.array(df.columns[3:]),
                is_train=(df["golub_split"] == "train").to_numpy(), sample_id=df["sample_id"].to_numpy(),
                source="Golub et al. 1999 (real data)")


def sigmoid(z):
    """Logistic function 1 / (1 + e^(−z)), written via tanh so that it never overflows."""
    return 0.5 * (1.0 + np.tanh(0.5 * np.asarray(z, float)))


def logsumexp(a, axis=None):
    """log(Σ exp(a)) computed stably (subtract the maximum first)."""
    a = np.asarray(a, float)
    m = np.max(a, axis=axis, keepdims=True)
    m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(a - m), axis=axis, keepdims=True)) + m
    return np.squeeze(out, axis=axis) if axis is not None else out.item()


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


def accuracy(y_true, y_pred) -> float:
    """Fraction of correct predictions."""
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def plot_decision_regions(predict, X, y, ax=None, resolution: int = 200, title: str = "",
                          class_names=None, pad: float = 0.5):
    """Shade the regions a 2-feature classifier assigns to each class and overlay the data.

    ``predict`` maps an (M, 2) array to M integer labels 0, 1, 2, …
    """
    X, y = np.asarray(X, float), np.asarray(y).astype(int)
    ax = ax or plt.gca()
    x0, x1 = X[:, 0].min() - pad, X[:, 0].max() + pad
    y0, y1 = X[:, 1].min() - pad, X[:, 1].max() + pad
    gx, gy = np.meshgrid(np.linspace(x0, x1, resolution), np.linspace(y0, y1, resolution))
    Z = np.asarray(predict(np.c_[gx.ravel(), gy.ravel()])).astype(int).reshape(gx.shape)
    k = max(int(y.max()), int(Z.max())) + 1
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(CLASS_COLORS[:k])
    ax.contourf(gx, gy, Z, levels=np.arange(k + 1) - 0.5, cmap=cmap, alpha=0.18)
    ax.contour(gx, gy, Z, levels=np.arange(k) + 0.5, colors="k", linewidths=0.8)
    for c in range(k):
        m = y == c
        if m.any():
            ax.scatter(X[m, 0], X[m, 1], s=18, color=CLASS_COLORS[c], edgecolor="k", linewidth=0.3,
                       label=(class_names[c] if class_names is not None else f"class {c}"))
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['fetch_course_file', 'simulate_golub_like', 'load_golub', 'sigmoid', 'logsumexp', 'standardize', 'accuracy', 'plot_decision_regions']))

---
## 1 · What is classification?

> 📍 **Where we are.** In Modules 06–07 the output $y$ was a number (a phenotype, a PSA level) and we learned a
> function $f$ with linear regression. Now the output is a **category**. Everything else — data matrix, training,
> likelihood, MLE/MAP — carries over.

A **classifier** is a function $f: X \to Y$ where $Y$ takes a few discrete values (slide 2): healthy/cancer,
ALL/AML, cell type 1/2/3. As in regression we work in two phases (slide 3):

1. **Training phase.** We are given input data $X$ *with* the correct outputs $Y$, and we learn $f$ — usually by
   learning a probability model $P(Y \mid X)$.
2. **Testing phase.** We apply $f$ to *previously unseen* inputs and compare its predictions with the truth. Only
   the testing phase tells us how well the classifier will do on the next patient.

**Notation.** As everywhere in this course, the data form a matrix $X$ with **N rows = samples** (patients,
individuals, cells) and **J columns = features** (genes, loci, markers); $y$ holds one class label per row, coded
0/1 for two classes (0 = healthy / ALL, 1 = cancer / AML). ⚠️ The slides change letters between sections: the k-NN
slides write the number of features as $J$; the naïve-Bayes slides write $n$ for the number of loci (features) and
$L$ for the number of training samples, with $X_i^\ell$ = feature $i$ of sample $\ell$. We follow the slides inside
each section and say so.

**Three kinds of classifier** (slide 4) — keep this table in mind for Modules 08–11:

| Classifier | Type | What it models | Learning |
|---|---|---|---|
| **k-nearest neighbours** | *non-parametric*: no model, no parameters | nothing — it stores the training data | none ("lazy") |
| **Naïve Bayes** | *parametric, generative* | the joint $P(Y, X \mid \theta) = P(Y)\,P(X \mid Y)$, then $P(Y\mid X)$ by Bayes' rule | closed-form MLE (counting) |
| **Logistic regression** ([Module 09](09_logistic_regression.ipynb)) | *parametric, discriminative* | $P(Y \mid X, \theta)$ directly | iterative (gradient ascent) |

Let us look at the real classification data set we will meet again in Section 10.

In [ ]:
# ── 1.1 A real classification data set: Golub et al. (1999) leukaemia expression ──
golub = load_golub()
print("Data source:", golub["source"])
Xg, yg, is_tr = golub["X"], golub["y"], golub["is_train"]
print(f"X: {Xg.shape[0]} patients × {Xg.shape[1]} genes (log10 expression);  y: 0 = ALL, 1 = AML")
for name, mask in [("training (original study)", is_tr), ("test (independent samples)", ~is_tr)]:
    print(f"  {name:27s}: {mask.sum():2d} patients = {np.sum(yg[mask] == 0)} ALL + {np.sum(yg[mask] == 1)} AML")

# The simplest possible 'classifier' ignores X and always predicts the most common training class.
majority = int(np.mean(yg[is_tr]) > 0.5)
print(f"\nBaseline: always predict '{'AML' if majority else 'ALL'}' → test accuracy "
      f"{accuracy(yg[~is_tr], np.full((~is_tr).sum(), majority)):.3f}")

🔍 **What just happened?** The data are a 72 × 3,571 matrix: each row is a patient, each column one gene's
(log10) expression level. Golub and colleagues designated 38 patients as the training set and 34 independently
collected patients as the test set — we keep exactly that split and never use the test patients for any choice, so
our test accuracies mean what they say. Note the
**baseline**: predicting "ALL" for everybody is already right for 20 of the 34 test patients (0.588). A classifier
is only interesting if it beats this number.

⚠️ **Accuracy can mislead.** With unbalanced classes a useless classifier can look decent. Module 09 adds
precision and recall (slide 36 of the logistic-regression deck).

### 📑 Slide connection — `003_classification-kNN-NB`, slides 2–4

- **What the slides show.** Classification = learning $f: X \to Y$ with discrete $Y$, with genomic and clinical examples (SNPs, expression, pathology images, medical records, TF binding sites); a training phase that learns $P(Y\mid X)$ and a testing phase on unseen data; and the three classifier types: k-NN (non-parametric, "lazy"), naïve Bayes (parametric, generative: models $P(Y, X\mid\theta)$), logistic regression (parametric, discriminative: models $P(Y\mid X, \theta)$).
- **What this notebook adds.** A real data set with a pre-registered train/test split, the majority-class baseline, and the course-wide N × J notation (and where the slides use other letters).
- **Exam tip.** Be able to say, for each of the three classifiers, *what is modelled* and *how it is learned*. "Generative" means a model of how $X$ is produced from $Y$ — not that it generates new data in practice.

---
## 2 · k-nearest neighbours: classification by resemblance

> 📍 **Where we are.** The first classifier of the lecture needs no probability model at all. It formalises the
> most natural idea in biology: *a new specimen probably belongs to the same group as the specimens it most
> resembles.*

**The algorithm** (slide 5). Given $N$ training points $\{\langle X_1, Y_1\rangle, \dots, \langle X_N, Y_N\rangle\}$,
perform **no learning**. To classify a new point $X_0$: find the $K$ training points closest to it and output the
**majority vote** of their labels.

🧮 **Math refresher — vectors and distance.** A sample with $J$ features is a **vector**
$X_i = (X_i^1, \dots, X_i^J)^\top$ — a column of $J$ numbers (the $^\top$, "transpose", turns a row into a column;
superscripts here index features, not powers). The squared **Euclidean distance** between two samples is

$$\lVert X_i - X_0\rVert^2 \;=\; (X_i - X_0)^\top (X_i - X_0) \;=\; \sum_{j=1}^{J} \big(X_i^j - X_0^j\big)^2 .$$

*In words:* subtract the two samples feature by feature, square each difference, add them up. In two dimensions
this is Pythagoras' theorem; in $J$ dimensions it is the same formula with more terms. The product
$a^\top b = \sum_j a_j b_j$ of a row with a column is the **dot product**, so the middle expression is just a
compact way of writing the sum.

In [ ]:
# ── 2.1 Euclidean distance between two cells, three ways ──
x_i = np.array([2.0, 1.0])      # a training cell: (marker A, marker B), log fluorescence
x_0 = np.array([3.5, 3.0])      # the new cell we want to classify

d2_loop = sum((x_i[j] - x_0[j]) ** 2 for j in range(len(x_i)))    # Σ_j (x_i^j − x_0^j)²
diff = x_i - x_0
d2_dot = diff @ diff                                                # (x_i − x_0)ᵀ(x_i − x_0)
d2_np = np.sum(diff ** 2)                                           # vectorised NumPy
print(f"squared distance: loop {d2_loop:.2f} | dot product {d2_dot:.2f} | np.sum {d2_np:.2f}")
print(f"distance ||x_i − x_0|| = {np.sqrt(d2_dot):.2f}   (a 1.5–2–2.5 right triangle)")

🔍 **What just happened?** The differences are 1.5 and 2.0, their squares 2.25 and 4.0, the squared distance 6.25
and the distance 2.5 — all three ways agree. k-NN only ever compares distances, so it does not matter whether we use
the distance or its square (the square root does not change the order); the squared version saves a square root.

Now the classifier itself. We write it once, carefully, and reuse it for the rest of the section.

In [ ]:
# ── 2.2 k-NN from scratch ──
def knn_predict(X_train, y_train, X_new, K, chunk=200):
    '''Majority vote among the K training samples closest (Euclidean) to each row of X_new.

    No learning happens: the 'model' is the training set itself. Ties are broken in favour of
    the smallest class label (np.argmax returns the first maximum).
    '''
    X_train, X_new = np.asarray(X_train, float), np.asarray(X_new, float)
    y_train = np.asarray(y_train, int)
    n_classes = y_train.max() + 1
    out = np.empty(len(X_new), int)
    for s in range(0, len(X_new), chunk):               # a few queries at a time keeps memory small
        q = X_new[s:s + chunk]
        d2 = np.zeros((len(q), len(X_train)))           # d2[a, b] = squared distance query a ↔ training b
        for j in range(X_train.shape[1]):
            d2 += (q[:, [j]] - X_train[:, j]) ** 2
        if K < len(X_train):
            nn = np.argpartition(d2, K - 1, axis=1)[:, :K]   # indices of the K smallest distances
        else:
            nn = np.tile(np.arange(len(X_train)), (len(q), 1))
        votes = np.stack([(y_train[nn] == c).sum(axis=1) for c in range(n_classes)], axis=1)
        out[s:s + chunk] = votes.argmax(axis=1)
    return out


# The picture of slide 5: five neighbours inside the circle (3 "−", 2 "+"), more "+" further away.
X_toy = np.array([[0.3, 0.8], [-0.6, 0.6], [-0.9, -0.3], [0.2, -0.7], [0.9, -0.5],   # inside the circle
                  [2.5, 2.0], [-2.4, 2.2], [2.6, -2.1]])                                # further away
y_toy = np.array([0, 0, 1, 0, 1, 1, 1, 1])                                             # 0 = "−", 1 = "+"
x_new = np.array([[0.0, 0.0]])
print("distances to X0:", np.round(np.sqrt(((X_toy - x_new) ** 2).sum(axis=1)), 2))
for K in (1, 5, 8):
    print(f"K = {K}: X0 is classified as '{'+−'[1 - knn_predict(X_toy, y_toy, x_new, K)[0]]}'")

🔍 **What just happened?** With $K = 5$ the circle of slide 5 contains three "−" and two "+", so $X_0$ is called
"−". The answer depends on $K$: the single nearest neighbour is also "−", but with $K = 8$ the five "+" outvote
the three "−". **K is a choice the user makes**, not something learned from the data — a *hyperparameter*.

### 2.1 Three cell types, two markers (SIMULATED)

To see decision boundaries like those on slides 6–10 we need data where we know the truth. Imagine a cytometry
experiment that measures two surface markers (A and B, log fluorescence) on immune cells of three types. Each cell
type consists of three sub-populations (for example resting, activated and dividing cells), so the classes overlap
and have irregular shapes. Because *we* simulated the data, we know the true class densities $p(x \mid c)$ and can
compute the best possible classifier — the **Bayes classifier** $\hat y = \arg\max_c P(c)\,p(x\mid c)$ (the purple
dashed line on slide 6, from the ISLR book). Its error rate, the **Bayes error**, is the floor no classifier can beat
on these data.

In [ ]:
# ── 2.3 Simulate three overlapping cell types ──
rng = np.random.default_rng(SEED + 2)
CENTRES = {0: [(1.0, 1.0), (2.8, 2.6), (0.6, 3.4)],      # sub-population centres of cell type 0
           1: [(3.0, 0.6), (1.6, 2.2), (4.4, 2.6)],      # … of cell type 1
           2: [(2.2, 4.2), (4.0, 4.0), (1.0, 4.8)]}      # … of cell type 2
SD = 0.65                                                # spread of every sub-population


def simulate_cells(n_per_class, rng):
    '''SIMULATED two-marker measurements: each class is an equal mixture of 3 Gaussian sub-populations.'''
    X, y = [], []
    for c, n in enumerate(n_per_class):
        comp = rng.integers(0, 3, n)                     # which sub-population each cell comes from
        X.append(np.array(CENTRES[c])[comp] + rng.normal(0, SD, (n, 2)))
        y.append(np.full(n, c))
    return np.vstack(X), np.concatenate(y)


def bayes_predict(Xq, class_prior):
    '''The Bayes classifier: argmax_c P(c) p(x | c), using the TRUE densities we simulated from.'''
    dens = np.zeros((len(Xq), 3))
    for c in range(3):
        for m in CENTRES[c]:
            dens[:, c] += np.exp(-np.sum((Xq - m) ** 2, axis=1) / (2 * SD ** 2)) / 3
    return np.argmax(dens * class_prior, axis=1)


N_PER = (120, 100, 80)                                   # cell types differ in abundance
X_tr, y_tr = simulate_cells(N_PER, rng)                  # 300 training cells
X_te, y_te = simulate_cells(tuple(10 * n for n in N_PER), rng)   # 3,000 test cells
PRIOR = np.array(N_PER) / sum(N_PER)
print(f"training cells: {len(y_tr)}, test cells: {len(y_te)};  class proportions {np.round(PRIOR, 2)}")
print(f"Bayes classifier test accuracy (the best possible): {accuracy(y_te, bayes_predict(X_te, PRIOR)):.3f}")

fig, ax = plt.subplots(figsize=(4.6, 4.0))
for c in range(3):
    ax.scatter(*X_tr[y_tr == c].T, s=14, color=CLASS_COLORS[c], edgecolor="k", lw=0.3, label=f"cell type {c}")
ax.set_xlabel("marker A (log fluorescence)"); ax.set_ylabel("marker B (log fluorescence)")
ax.set_title("SIMULATED: 300 training cells, 3 overlapping types"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The three cell types overlap: in the middle of the plot all three colours mix. Even
the Bayes classifier, which knows the true densities, gets only 78 % of the test cells right — more than one cell
in five sits in a region where another type is more likely. That is the **irreducible error** of this problem; a
classifier that does much better on *training* data is fitting noise.

### 2.2 Decision boundaries and the choice of K (slides 6–11)

A **decision boundary** separates the regions of feature space that the classifier assigns to different classes.
We shade the regions for $K = 1$, $K = 15$ and $K = N$ and overlay the Bayes boundary (purple, dashed).

In [ ]:
# ── 2.4 Decision regions for K = 1, 15 and N (slides 6–11) ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.9))
for ax, K in zip(axes, [1, 15, len(y_tr)]):
    plot_decision_regions(lambda Q, K=K: knn_predict(X_tr, y_tr, Q, K), X_tr, y_tr, ax=ax, resolution=150,
                          class_names=["type 0", "type 1", "type 2"])
    gx, gy = np.meshgrid(np.linspace(*ax.get_xlim(), 150), np.linspace(*ax.get_ylim(), 150))
    zb = bayes_predict(np.c_[gx.ravel(), gy.ravel()], PRIOR).reshape(gx.shape)
    ax.contour(gx, gy, zb, levels=[0.5, 1.5], colors="purple", linestyles="--", linewidths=1.2)
    acc_tr = accuracy(y_tr, knn_predict(X_tr, y_tr, X_tr, K))
    acc_te = accuracy(y_te, knn_predict(X_tr, y_tr, X_te, K))
    ax.set_title(f"K = {K}: train acc {acc_tr:.2f}, test acc {acc_te:.2f}")
    ax.set_xlabel("marker A"); ax.set_ylabel("marker B")
axes[0].legend(fontsize=7, loc="lower right")
plt.suptitle("k-NN decision regions (black) vs the Bayes boundary (purple dashed) — SIMULATED", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** These are the three regimes of slides 10–11:

- **K = 1** draws a **wiggly** boundary with islands around individual cells. Every training cell is its own
  nearest neighbour, so training accuracy is a perfect 1.00 — but test accuracy is only 0.69: the boundary
  follows the noise ("small K: classification is sensitive to the noise").
- **K = 15** gives a **smoother** boundary that tracks the purple Bayes boundary reasonably well, and its test
  accuracy (0.76) is close to the Bayes optimum (0.78).
- **K = N = 300**: every query has the *same* 300 neighbours — the whole training set — so every cell receives the
  same label, the most common training class (type 0). Test accuracy drops to 0.40, the fraction of type-0 cells:
  "if K = N, all test inputs will receive the same classification".

Let us scan K systematically.

In [ ]:
# ── 2.5 Training and test accuracy as a function of K ──
Ks = [1, 2, 3, 5, 8, 12, 15, 20, 30, 45, 60, 80, 100, 150, 200, 300]
acc_train = [accuracy(y_tr, knn_predict(X_tr, y_tr, X_tr, K)) for K in Ks]
acc_test = [accuracy(y_te, knn_predict(X_tr, y_tr, X_te, K)) for K in Ks]
bayes_acc = accuracy(y_te, bayes_predict(X_te, PRIOR))
for K, a, b in zip(Ks, acc_train, acc_test):
    print(f"K = {K:3d}: train {a:.3f}  test {b:.3f}")

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(Ks, 1 - np.array(acc_train), "o-", color="tab:blue", label="training error")
ax.plot(Ks, 1 - np.array(acc_test), "s-", color="tab:orange", label="test error")
ax.axhline(1 - bayes_acc, color="purple", ls="--", lw=1, label="Bayes error (best possible)")
ax.set_xscale("log"); ax.set_xlabel("K (number of neighbours, log scale)"); ax.set_ylabel("error rate")
ax.set_title("small K overfits, large K over-smooths (SIMULATED)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Training error starts at zero for K = 1 and rises with K. Test error has the typical
**U shape**: it is high for K = 1 (noise), lowest over a broad range of intermediate K (here roughly 5–60,
within about two percentage points of the Bayes error), and climbs again for large K (over-smoothing), jumping to 0.60 at K = N.
This is the first appearance of the **bias–variance trade-off** that runs through the whole course: small K =
flexible, low bias, high variance; large K = rigid, high bias, low variance.

⚠️ **Do not choose K with the test set.** We may look at the test curve here only because the data are simulated
and the test set is huge — it is a teaching device. In a real study K is chosen on the training data alone, with a
validation set or cross-validation ([Module 23](23_model_selection.ipynb)); the test set is opened once, at the end.

✏️ **Your turn.** Change `N_PER` to `(1200, 1000, 800)` (ten times more training cells) and re-run cells 2.3–2.5.
How do the best K and the best test error change? (Hint: with more data, larger K can average away noise without
smoothing over real structure, and the best test error moves closer to the Bayes error.)

### 2.3 Distances need comparable units: feature scaling

k-NN has no parameters that could adapt to the units of the features — it simply adds squared differences. What
happens if marker B is recorded on a 100× larger scale (another instrument gain, or raw instead of normalised units)?

In [ ]:
# ── 2.6 Feature scaling changes k-NN completely ──
scale = np.array([1.0, 100.0])                     # marker B now in units 100× larger
Xs_tr, Xs_te = X_tr * scale, X_te * scale
acc_orig = accuracy(y_te, knn_predict(X_tr, y_tr, X_te, 15))
acc_scaled = accuracy(y_te, knn_predict(Xs_tr, y_tr, Xs_te, 15))
acc_B_only = accuracy(y_te, knn_predict(X_tr[:, [1]], y_tr, X_te[:, [1]], 15))
Z_tr, mu_tr, sd_tr = standardize(Xs_tr)            # z-scores using TRAINING means and sds …
Z_te, _, _ = standardize(Xs_te, mu_tr, sd_tr)      # … applied unchanged to the test cells
acc_z = accuracy(y_te, knn_predict(Z_tr, y_tr, Z_te, 15))
print(f"K = 15 test accuracy, original units          : {acc_orig:.3f}")
print(f"K = 15 test accuracy, marker B ×100           : {acc_scaled:.3f}")
print(f"K = 15 test accuracy, marker B alone          : {acc_B_only:.3f}")
print(f"K = 15 test accuracy, ×100 then standardised  : {acc_z:.3f}")

🔍 **What just happened?** Multiplying one marker by 100 makes the squared distance almost entirely a marker-B
distance, so k-NN effectively ignores marker A: its accuracy falls to about what marker B achieves alone. Z-scoring
each feature (subtract the mean, divide by the standard deviation — Module 03) puts both markers on the same scale
and restores the original performance.

⚠️ **Two pitfalls.** (1) Always ask whether your features are on comparable scales before using any distance-based
method (k-NN here, k-means and hierarchical clustering in Module 12, SVM kernels in Module 11). (2) Compute the means
and standard deviations on the **training** data only and apply them unchanged to test data — otherwise information
from the test set leaks into training.

### 2.4 No training, expensive testing (slide 12)

k-NN is "lazy": training is free (store the data), but every prediction must compute the distance to **every**
training sample. Let us time it.

In [ ]:
# ── 2.7 Test-time cost grows with the number of training samples ──
rng_t = np.random.default_rng(SEED + 27)
X_query, _ = simulate_cells((200, 200, 100), rng_t)        # 500 new cells to classify
sizes, seconds = [1_000, 4_000, 16_000, 64_000], []
for n in sizes:
    X_big, y_big = simulate_cells((n * 4 // 10, n // 3, n - n * 4 // 10 - n // 3), rng_t)
    best = np.inf
    for _ in range(3):                                      # repeat and keep the fastest run (less timing noise)
        t0 = time.perf_counter()
        knn_predict(X_big, y_big, X_query, K=15)
        best = min(best, time.perf_counter() - t0)
    seconds.append(best)
for n, s in zip(sizes, seconds):
    print(f"N = {n:6,d} training cells: {s:6.3f} s for 500 predictions ({1e3 * s / 500:.3f} ms each) = "
          f"{1e9 * s / (500 * n):5.1f} ns per query–training-cell distance")

🔍 **What just happened?** Your exact timings depend on your computer (and on what else it is doing), but the
pattern does not: the cost *per distance computed* (last column) stays roughly constant — within a factor of two
or three (fixed overheads matter for small N, memory traffic for large N) — so the total prediction time grows in
proportion to N: 64 times more training cells, tens of times more waiting. The cost of k-NN is paid at **test time** and grows linearly with N. For a single-cell atlas with
millions of reference cells, people therefore use clever data structures (k-d trees, approximate nearest-neighbour
search) — the idea of k-NN stays the same.

✏️ **Your turn.** k-NN has no probability model, but you can still get a "probability": the fraction of the K
neighbours that belong to each class. Modify `knn_predict` to return `votes / K` and look at the vote fractions for
a few test cells near a boundary. Why are these fractions coarse (multiples of 1/K)?

### 📑 Slide connection — `003_classification-kNN-NB`, slides 5–12

- **What the slides show.** k-NN performs no learning: classify $X_0$ by the majority vote of its K nearest training samples under the squared Euclidean distance $(X_i - X_0)^\top(X_i - X_0)$ (slide 5, a 5-NN picture). Slides 6–10 show 3-class decision boundaries (from ISLR): 1-NN is wiggly, 15-NN smoother; the purple dashed line is the Bayes classifier. Slide 11: small K is noise-sensitive, large K over-smooths, K = N gives every input the same class. Slide 12: no training, but each prediction visits every training sample.
- **What this notebook adds.** A from-scratch implementation; a simulation in which the Bayes boundary and Bayes error are known exactly; the U-shaped test-error curve; the feature-scaling pitfall; a timing experiment; and the warning that K must not be tuned on the test set.
- **Exam tip.** Explain *why* 1-NN has zero training error (each point is its own nearest neighbour) and why that says nothing about test error. Know what happens at K = N, and that k-NN's cost is at prediction time.

---
## 3 · Given my genotypes, am I going to develop cancer?

> 📍 **Where we are.** k-NN needs no model, but it gives no insight into *why* a sample is classified as it is, its
> "probabilities" are crude, and it must keep all the training data. The rest of the notebook builds a
> **probabilistic** classifier step by step, following the slides' genotype example exactly.

The slides' training data (slide 14) are six individuals genotyped at three loci, with their health outcome.
Genotypes are coded as on the slides — **0 = AA** (homozygous for the minor allele), **1 = AT** (heterozygous),
**2 = TT** (homozygous for the major allele) — i.e. the number of T alleles; $Y = 1$ means cancer, $Y = 0$ healthy.

In [ ]:
# ── 3.1 The six individuals of slide 14 ──
GENO = {0: "AA", 1: "AT", 2: "TT"}
X_gen, y_gen = NB_GENOTYPE_TABLE[:, :3], NB_GENOTYPE_TABLE[:, 3]
show = pd.DataFrame(index=pd.Index(range(1, 7), name="individual"))
for i in range(3):
    show[f"locus {i + 1} (X{i + 1})"] = [f"{v} = {GENO[v]}" for v in X_gen[:, i]]
show["outcome Y"] = np.where(y_gen == 1, "1 = cancer", "0 = healthy")
print(show.to_string())
print("\nX has shape", X_gen.shape, "(L = 6 individuals × n = 3 loci); y has shape", y_gen.shape)

We want a probability model $P(Y \mid X_1, X_2, X_3)$. The most direct idea (slide 16) is a **lookup table**: one
row per possible genotype combination, holding $P(Y = 1 \mid X_1, X_2, X_3)$ (and $P(Y = 0 \mid \dots) = 1 - P(Y=1 \mid
\dots)$). The slide shows the first rows of such a table with illustrative values (0.01, 0.50, 0.30, 0.25, 0.70,
0.05 for combinations 000 to 012). How many rows are there, and how many could our six individuals fill?

In [ ]:
# ── 3.2 How much data does each row of the full table P(Y | X1, X2, X3) get? ──
combos = list(itertools.product([0, 1, 2], repeat=3))          # all 3 × 3 × 3 = 27 genotype combinations
rows = []
for x in combos:
    match = np.all(X_gen == x, axis=1)                          # individuals with exactly this genotype
    rows.append({"X1": x[0], "X2": x[1], "X3": x[2], "individuals": int(match.sum()),
                 "cancer cases": int(y_gen[match].sum())})
full_table = pd.DataFrame(rows)
print(f"{len(combos)} genotype combinations; observed at least once: {(full_table.individuals > 0).sum()}; "
      f"never observed: {(full_table.individuals == 0).sum()}")
print("\nThe observed rows (each estimate of P(Y=1 | x) would rest on ONE person):")
print(full_table[full_table.individuals > 0].to_string(index=False))

🔍 **What just happened?** With three loci there are already 27 genotype combinations. Our six individuals occupy
six of them — one person each — and 21 rows have **no data at all**. Learning the table from counts would give
$P(Y = 1 \mid x) = 1$ or $0$ for the six observed rows (one person each!) and "no idea" for everyone else. The table is
far too flexible for the data.

### 3.1 Counting parameters (slides 18 and 20)

🧮 **Math refresher — powers and counting.** If each locus has 3 possible genotypes, two loci have $3 \times 3 = 9$
combinations, three loci $3^3 = 27$, and $n$ loci $3^n$ ("3 to the power n": $n$ factors of 3). Powers grow
explosively: every extra locus *triples* the count.

- **Full conditional table** (slide 18): one free number $P(Y = 1 \mid x)$ per combination ⇒ $3^n$ parameters.
  For $n = 30$ SNPs, $3^{30} \approx 2 \times 10^{14}$.
- **Bayes' rule without further assumptions** (slide 20): write
  $P(Y \mid X) = P(X \mid Y)\,P(Y) / P(X)$ and learn the two pieces of the numerator instead.

$$P(Y \mid X_1,\dots,X_n) \;=\; \frac{P(X_1,\dots,X_n \mid Y)\;P(Y)}{P(X_1,\dots,X_n)}$$

*In words:* the probability of cancer given the genotypes equals how typical these genotypes are among cancer
cases, times how common cancer is, divided by how common the genotypes are overall. For each class, the
distribution $P(X_1, \dots, X_n \mid Y = j)$ assigns a probability to each of the $3^n$ combinations; they must sum to
1, so only $3^n - 1$ are free. Two classes give $(3^n - 1) \times 2$, and $P(Y)$ adds one more (slide 28):
$(3^n - 1) \times 2 + 1$ — even *more* than the full table. Bayes' rule alone does not help (slide 20: "Still too
many parameters!").

In [ ]:
# ── 3.3 Parameter counts of the three representations (slides 18, 20, 23, 28) ──
def n_parameters(n: int) -> dict:
    '''Free parameters for P(Y | X1..Xn) with n three-valued loci and a binary outcome Y.'''
    return {"full table P(Y|X)": 3 ** n,                       # slide 18
            "Bayes rule, no assumption": (3 ** n - 1) * 2 + 1,  # slides 20 and 28
            "naive Bayes": 2 * n * 2 + 1}                       # slides 23 and 28


counts = pd.DataFrame({n: n_parameters(n) for n in [1, 2, 3, 10, 20, 30]}).T
counts.index.name = "n loci"
print(counts.to_string(formatters={c: "{:,}".format for c in counts.columns}))
print(f"\n3^30 = {3 ** 30:,} ≈ {3 ** 30:.1e}  (slide 18: '3^30 ~ 2x10^14')")
print(f"World population ≈ 8,000,000,000 → {3 ** 30 / 8e9:,.0f} combinations per living human")

🔍 **What just happened?** For $n = 3$ the full table needs 27 numbers and the Bayes-rule version 53. For $n = 30$
loci both are in the hundreds of trillions — there are roughly 26,000 genotype combinations *per living human*, so
most combinations will never be observed even once, let alone often enough to estimate a probability. The last
column is the punchline of the next section: **naïve Bayes needs 4n + 1 parameters — 121 for 30 loci**.

---
## 4 · Conditional independence: the assumption that makes learning possible

> 📍 **Where we are.** Bayes' rule alone did not reduce the count because $P(X_1, \dots, X_n \mid Y)$ is still a table over
> all $3^n$ combinations. We need an assumption about *how the loci relate to each other* (slides 22–23).

🧮 **Math refresher — conditional probability and the chain rule** (Module 02). $P(A \mid B) = P(A, B)/P(B)$ is the
probability of $A$ among the cases where $B$ holds. Rearranged, $P(A, B) = P(A \mid B)\,P(B)$ — the **chain rule**.
Everything can additionally be conditioned on $Y$ (just restrict attention to one class):

$$P(X_1, X_2 \mid Y) \;=\; P(X_1 \mid X_2, Y)\; P(X_2 \mid Y) \qquad \text{(chain rule, always true)}$$

*In words:* the chance of seeing both genotypes in a cancer patient = the chance of the second genotype in a cancer
patient × the chance of the first given the second (and cancer).

$X_1$ and $X_2$ are **conditionally independent given $Y$** if, once we know $Y$, learning $X_2$ tells us nothing more
about $X_1$: $P(X_1 \mid X_2, Y) = P(X_1 \mid Y)$. Then (slide 22)

$$P(X_1, X_2 \mid Y) \;=\; P(X_1 \mid Y)\; P(X_2 \mid Y).$$

*In words:* within each class, the joint probability of two genotypes is the product of their separate
probabilities. **Naïve Bayes** assumes this for every pair of features (slide 23):

$$P(X_1, \dots, X_n \mid Y) \;=\; \prod_{i=1}^{n} P(X_i \mid Y).$$

*In words:* within the cancer patients (and separately within the healthy individuals), the loci behave like
independent dice. The symbol $\prod_{i=1}^n$ means "multiply the terms for $i = 1, 2, \dots, n$" — the product
analogue of $\sum$.

⚠️ **A common misconception:** naïve Bayes does **not** assume the features are independent. It assumes they are
independent **within each class**. Features can be strongly correlated overall *because* they all depend on $Y$. The
next experiment shows both facts at once.

In [ ]:
# ── 4.1 Conditional independence is not independence (SIMULATED population) ──
rng = np.random.default_rng(SEED + 4)
L_sim = 20_000
y_sim = (rng.random(L_sim) < 0.5).astype(int)
# P(X_i = k | Y = j) for two loci: genotype frequencies differ between cases (1) and controls (0)
FREQ = {1: [[0.6, 0.3, 0.1], [0.1, 0.6, 0.3]],
        0: [[0.1, 0.3, 0.6], [0.5, 0.3, 0.2]]}
X_sim = np.zeros((L_sim, 2), int)
for j in (0, 1):                                   # the naive-Bayes generative story: Y first …
    m = y_sim == j
    for i in range(2):                             # … then each locus independently given Y
        X_sim[m, i] = rng.choice(3, size=m.sum(), p=FREQ[j][i])


def joint_table(a, b):
    '''3 × 3 table of the empirical joint probability P(A = r, B = c).'''
    t = np.zeros((3, 3))
    np.add.at(t, (a, b), 1)
    return t / len(a)


m1 = y_sim == 1
J_c = joint_table(X_sim[m1, 0], X_sim[m1, 1])                    # P(X1, X2 | Y = 1)
P_c = np.outer(J_c.sum(axis=1), J_c.sum(axis=0))                  # P(X1 | Y = 1) P(X2 | Y = 1)
J_all = joint_table(X_sim[:, 0], X_sim[:, 1])                    # P(X1, X2), classes pooled
P_all = np.outer(J_all.sum(axis=1), J_all.sum(axis=0))            # P(X1) P(X2)
print("Within cancer cases:  max |P(X1,X2|Y=1) − P(X1|Y=1)P(X2|Y=1)| =", np.abs(J_c - P_c).max().round(4))
print("Everybody pooled:     max |P(X1,X2)     − P(X1)P(X2)|         =", np.abs(J_all - P_all).max().round(4))
print("\nP(X1 = 0, X2 = 0) pooled:", J_all[0, 0].round(4), " vs  P(X1 = 0)·P(X2 = 0):", P_all[0, 0].round(4))

🔍 **What just happened?** We generated each locus independently *within* each class — exactly the naïve-Bayes
story. Inside the cancer cases, the joint table matches the product of the marginals up to sampling noise (largest
difference 0.003). Pooling everybody, the two loci are clearly **dependent** (largest difference 0.05): "AA at both
loci" occurs in 5.2 % of people, only half of the 10.3 % expected if the loci were independent. The reason is the
shared cause $Y$: AA at locus 1 points towards "case", and cases rarely carry AA at locus 2. Conditional
independence and independence are different properties.

**Biologically**, when is the assumption reasonable? For loci on different chromosomes in a genetically homogeneous
population, genotypes are close to independent within cases and within controls if each locus acts on its own. It
fails for **neighbouring loci in linkage disequilibrium** (they are inherited together — Section 9), for
populations with **structure** (ancestry links many loci — Module 14), and for genes in the same pathway
(co-expression — Section 10).

### 4.1 Counting parameters under naïve Bayes (slides 23 and 28)

With the assumption we only need, for every locus $i$ and every class $j$, the three probabilities
$P(X_i = 0 \mid Y = j)$, $P(X_i = 1 \mid Y = j)$, $P(X_i = 2 \mid Y = j)$. They sum to 1, so **2 are free**. That is
$2 \times 2 = 4$ numbers per locus ("2 × 2" on slide 23), $2 \cdot n \cdot 2$ for $n$ loci, plus 1 for $P(Y)$:

$$\#\theta_{\text{naive Bayes}} \;=\; 2 \cdot n \cdot 2 + 1 \;=\; 4n + 1 \qquad\text{vs}\qquad \#\theta_{\text{no assumption}} = (3^n - 1)\times 2 + 1 .$$

*In words:* the cost grows **linearly** in the number of loci instead of exponentially. (The course summary
abbreviates this as "3ⁿ → 2n".)

In [ ]:
# ── 4.2 Exponential versus linear growth of the parameter count ──
ns = np.arange(1, 31)
fig, ax = plt.subplots(figsize=(6, 3.4))
styles = {"full table P(Y|X)": ("tab:red", "-"), "Bayes rule, no assumption": ("tab:orange", "--"),
          "naive Bayes": ("tab:blue", "-")}
for key, (col, ls) in styles.items():
    ax.plot(ns, [float(n_parameters(n)[key]) for n in ns], color=col, ls=ls, lw=2, label=key)
ax.axhline(8e9, color="gray", lw=1, ls=":")
ax.text(1, 1.6e10, "≈ number of humans alive", fontsize=7, color="gray")
ax.set_yscale("log"); ax.set_xlabel("number of loci n"); ax.set_ylabel("free parameters (log scale)")
ax.set_title("conditional independence buys tractability"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
print({n: n_parameters(n)["naive Bayes"] for n in (3, 30, 500_000)})

🔍 **What just happened?** On a logarithmic axis an exponential is a straight line and a linear function an almost
flat curve. The two exponential representations cross the world population at about 21 loci; naïve Bayes needs 13
parameters for the three loci of our table, 121 for 30 loci, and only two million for a 500,000-SNP chip. This is
the second of the three ideas that recur in the course: **conditional independence buys tractability** — we will see
it again as 31 → 10 parameters in the alarm Bayesian network (Module 15), the chain structure of hidden Markov models
(Module 19) and the sparse precision matrix of Gaussian graphical models (Module 18).

---
## 5 · The full naïve Bayes model (slides 24–32)

> 📍 **Where we are.** We have the assumption; now we write the model down completely, with every parameter named —
> step 1 of the course's two-step template (*specify a probability model*; step 2, *estimate its parameters*, is
> Section 6).

Putting Bayes' rule and conditional independence together (slide 24):

$$P(Y \mid X_1, \dots, X_n) \;=\; \frac{P(Y)\,\prod_{i=1}^{n} P(X_i \mid Y)}{P(X_1, \dots, X_n)} .$$

*In words:* prior probability of the class × how typical each genotype is in that class, divided by a normalising
constant that makes the two class probabilities add up to 1.

**The class prior is a Bernoulli distribution** with one parameter $\pi = P(Y = 1)$:

$$P(Y) \;=\; \pi^{Y}\,(1-\pi)^{1-Y} .$$

*In words:* if $Y = 1$ this is $\pi^1(1-\pi)^0 = \pi$; if $Y = 0$ it is $\pi^0(1-\pi)^1 = 1-\pi$. (Recall $a^0 = 1$ and
$a^1 = a$: the exponent acts as an on/off switch.)

**Each genotype given the class is a multinoulli (categorical) distribution** — a three-sided die — with parameters
$\theta_{ijk} = P(X_i = k \mid Y = j)$, written with the **indicator function** $\mathbb{I}[X_i = k]$ (1 if the
statement in brackets is true, 0 otherwise):

$$P(X_i \mid Y = j) \;=\; \prod_{k=0}^{2} \theta_{ijk}^{\;\mathbb{I}[X_i = k]}, \qquad \theta_{ij0} + \theta_{ij1} + \theta_{ij2} = 1 .$$

*In words:* the same on/off trick with three switches — exactly one indicator is 1, so the product picks out the
probability of the genotype actually observed. For example (slide 25), if $X_i = 0$:
$\theta_{ij0}^{1}\theta_{ij1}^{0}\theta_{ij2}^{0} = \theta_{ij0}$.

**The denominator** (slide 26) is not a new parameter. By the law of total probability the genotypes occur either
with $Y = 1$ or with $Y = 0$:

$$P(X_1,\dots,X_n) \;=\; P(Y{=}1)\prod_{i} P(X_i \mid Y{=}1) \;+\; P(Y{=}0)\prod_{i} P(X_i \mid Y{=}0).$$

*In words:* evaluate the numerator for both classes and add them up.

**The parameters** (slide 27): $\theta = \{\pi, \theta_{ijk}\}$ for the two classes $j \in \{0, 1\}$, the $n$ loci
$i = 1, \dots, n$ (the slide prints "$i = 0, \dots, n$" — a typo; there are $n$ loci) and the three genotypes
$k \in \{0, 1, 2\}$. In code we store them as an array `theta[j, i, k]` (with Python's 0-based locus index).

In [ ]:
# ── 5.1 The building blocks, written exactly as on slides 24–25 ──
def bernoulli_pmf(y, pi):
    '''P(Y = y) = π^y (1 − π)^(1 − y).'''
    return pi ** y * (1 - pi) ** (1 - y)


def categorical_pmf(x, theta_row):
    '''P(X = x) = Π_k θ_k^{I[x = k]} — the indicator exponent switches on exactly one factor.'''
    return np.prod([theta_row[k] ** int(x == k) for k in range(len(theta_row))])


theta_demo = np.array([0.2, 0.5, 0.3])        # an illustrative three-sided die: P(AA), P(AT), P(TT)
print("Bernoulli with π = 0.3:  P(Y=1) =", bernoulli_pmf(1, 0.3), "  P(Y=0) =", bernoulli_pmf(0, 0.3))
for x in range(3):
    print(f"categorical: P(X = {x}) = {categorical_pmf(x, theta_demo)}  (should be θ_{x} = {theta_demo[x]})")

In [ ]:
# ── 5.2 Joint probability, posterior and the classification rule (slides 26, 30–32) ──
def nb_joint(x, pi, theta):
    '''[P(Y = 0, X = x), P(Y = 1, X = x)] = P(Y = j) · Π_i P(X_i = x_i | Y = j)  (slide 31).'''
    x = np.asarray(x)
    return np.array([bernoulli_pmf(j, pi) * np.prod(theta[j, np.arange(len(x)), x]) for j in (0, 1)])


def nb_posterior(x, pi, theta):
    '''P(Y = 1 | X = x): the Y = 1 numerator divided by the sum over both classes (slide 26).'''
    joint = nb_joint(x, pi, theta)
    return joint[1] / joint.sum() if joint.sum() > 0 else np.nan     # 0/0 is undefined → NaN


# Sanity check: if every locus has the same genotype distribution in both classes, the genotypes carry no
# information and the posterior must equal the prior.
theta_flat = np.full((2, 3, 3), 1 / 3)
print("uninformative loci, prior π = 0.3 → posterior for genotype (0, 2, 1):",
      round(nb_posterior([0, 2, 1], 0.3, theta_flat), 6))

🔍 **What just happened?** The indicator products pick the right $\theta$ for each genotype, and our posterior
function passes a basic sanity check: uninformative features leave the prior unchanged (0.3 in, 0.3 out).

**Classifying a new individual** $X^{new} = (x_1, \dots, x_n)$ follows three steps (slides 30–32):

1. compute $P(Y = 1, x_1, \dots, x_n) = \pi \prod_i \theta_{i1x_i}$;
2. compute $P(Y = 0, x_1, \dots, x_n) = (1 - \pi) \prod_i \theta_{i0x_i}$;
3. predict the class with the larger value.

The denominator is the same for both classes, so it is **not needed for the decision** — only for reporting a
probability. (Slide 30 writes steps 1–2 with the full posterior; slides 31–32 show the "simpler method" with the
joint, which gives the same decision.)

### 📑 Slide connection — `003_classification-kNN-NB`, slides 14–28 (and 30–32)

- **What the slides show.** The six-individual genotype table (0 = AA minor homozygous, 1 = AT, 2 = TT major homozygous); a full lookup table for $P(Y\mid X)$ needs $3^n$ numbers ($3^{30}\approx 2\times10^{14}$, slide 18); Bayes' rule alone needs $(3^n-1)\times 2$ for $P(X\mid Y)$ plus 1 for $P(Y)$ (slide 20); conditional independence $P(X_1\ldots X_n\mid Y)=\prod_i P(X_i\mid Y)$ gives $2\times n\times 2$ (slide 23) — "Success!"; the full model with a Bernoulli $P(Y)=\pi^Y(1-\pi)^{1-Y}$, multinoulli $P(X_i\mid Y=j)=\prod_k\theta_{ijk}^{\mathbb{I}[X_i=k]}$ and the denominator obtained by summing the numerator over $Y$ (slides 24–27); the counts $2\cdot n\cdot 2+1$ vs $(3^n-1)\cdot 2+1$ (slide 28); the three-step classification rule (slides 30–32).
- **What this notebook adds.** An enumeration showing that 21 of the 27 rows of the full table have no data; exact parameter counts up to 30 loci and a log-scale plot; a simulation showing that conditional independence is *not* independence; the indicator and Bernoulli formulas as runnable code; and a note that slide 27's "$i = 0,\ldots,n$" should read $i = 1,\ldots,n$.
- **Exam tip.** Parameter counting is a classic exam question. Count *free* parameters: a distribution over $m$ values has $m-1$. Be ready to redo the count for other cases (binary features: $2n+1$; $k$ classes and $m$-valued features: $k\,n\,(m-1) + (k-1)$).

---
## 6 · Learning the parameters: maximum likelihood is counting (slides 34–46)

> 📍 **Where we are.** Step 2 of the template: estimate $\theta = \{\pi, \theta_{ijk}\}$ from the training data. In
> Module 05 you met the maximum-likelihood estimate (MLE) of a coin; naïve Bayes is "many coins and dice at once".

**Which likelihood?** (slide 34). In linear regression we maximised the **conditional** likelihood
$\prod_{\ell} P(Y^\ell \mid X^\ell)$ (MCLE): $X$ was treated as given. Naïve Bayes is *generative* — it models how
$X$ is produced — so we maximise the likelihood of the **whole data**, inputs and outputs (MLE):

$$\hat\theta_{MLE} \;=\; \arg\max_{\pi,\theta}\; \log \prod_{\ell=1}^{L} P(Y^\ell, X^\ell) \;=\; \arg\max_{\pi,\theta}\; \log \prod_{\ell=1}^{L} P(Y^\ell)\prod_{i=1}^{n} P(X_i^\ell \mid Y^\ell).$$

*In words:* choose the parameters under which the six observed individuals — genotypes *and* outcomes — were most
probable. The second equality uses $P(Y, X) = P(Y)P(X \mid Y)$ and the conditional-independence assumption.

🧮 **Math refresher — logs turn products into sums.** $\log(ab) = \log a + \log b$, so $\log \prod_\ell a_\ell =
\sum_\ell \log a_\ell$. The log is increasing, so maximising the log-likelihood gives the same $\hat\theta$ as
maximising the likelihood. Applying this twice (slides 35–38):

$$\log \prod_{\ell} P(Y^\ell)\prod_{i} P(X_i^\ell \mid Y^\ell) \;=\; \underbrace{\sum_{\ell=1}^{L} \log P(Y^\ell)}_{\text{depends only on } \pi} \;+\; \sum_{i=1}^{n}\underbrace{\sum_{\ell=1}^{L} \log P(X_i^\ell \mid Y^\ell)}_{\text{depends only on } \theta_{i\cdot\cdot}} .$$

*In words:* the log-likelihood splits into separate pieces — one for the class prior and one for each locus — that
share no parameters. Maximising the sum means maximising **each piece on its own**: $n + 1$ small, independent
problems instead of one big one. This decomposition is why naïve Bayes needs no iterative optimisation.

**The prior piece is a coin** (slide 40). With $n_1 = \#(Y = 1)$ and $n_0 = \#(Y = 0)$:
$\sum_\ell \log \pi^{Y^\ell}(1-\pi)^{1-Y^\ell} = n_1 \log \pi + n_0 \log(1-\pi)$. Setting the derivative to zero
(Module 04):

$$\frac{n_1}{\pi} - \frac{n_0}{1-\pi} = 0 \quad\Longrightarrow\quad \hat\pi = \frac{n_1}{n_1 + n_0} = \frac{\#(\text{samples with } Y=1)}{\#(\text{samples})}.$$

*In words:* the best estimate of the cancer probability is the fraction of cancer cases. For the slide's coin
(H, H, T, T, H, H, T) this gives $\hat q = 4/7$.

**Each locus piece is a die.** Among the $N_j$ samples of class $j$, let $n_k$ be the number with $X_i = k$. The
piece is $n_0 \log\theta_0 + n_1\log\theta_1 + n_2\log\theta_2$ with $\theta_2 = 1 - \theta_0 - \theta_1$. Its
partial derivatives (derivative with respect to one variable, holding the other fixed) are

$$\frac{\partial}{\partial \theta_0}: \;\frac{n_0}{\theta_0} - \frac{n_2}{\theta_2} = 0, \qquad \frac{\partial}{\partial \theta_1}:\; \frac{n_1}{\theta_1} - \frac{n_2}{\theta_2} = 0 .$$

*In words:* at the optimum, $n_0/\theta_0 = n_1/\theta_1 = n_2/\theta_2$, i.e. each probability is proportional to
its count. Because they must add up to 1:

$$\hat\theta_{ijk} \;=\; P(X_i = k \mid Y = j) \;=\; \frac{\#(\text{samples with } X_i = k,\; Y = j)}{\#(\text{samples with } Y = j)}\qquad\text{(slide 41).}$$

*In words:* among the individuals of class $j$, count how often locus $i$ has genotype $k$. That is all.
Let us check both results numerically before using them.

In [ ]:
# ── 6.1 Numerical check: the log-likelihood peaks at the count ratios ──
q = np.linspace(0.001, 0.999, 999)
heads, tails = 4, 3                                          # the coin of slide 40: H, H, T, T, H, H, T
loglik_coin = heads * np.log(q) + tails * np.log(1 - q)
print(f"coin: grid maximum at q = {q[np.argmax(loglik_coin)]:.3f};  count ratio 4/7 = {4 / 7:.3f}")

n_k = np.array([5, 3, 2])                                    # counts of genotypes 0, 1, 2 in one class
g = np.arange(1, 100) / 100
T0, T1 = np.meshgrid(g, g, indexing="ij")
T2 = 1 - T0 - T1                                             # the constraint θ0 + θ1 + θ2 = 1
ll = np.full(T0.shape, -np.inf)
ok = T2 > 1e-9
ll[ok] = n_k[0] * np.log(T0[ok]) + n_k[1] * np.log(T1[ok]) + n_k[2] * np.log(T2[ok])
a, b = np.unravel_index(np.argmax(ll), ll.shape)
print(f"die:  grid maximum at θ = ({T0[a, b]:.2f}, {T1[a, b]:.2f}, {T2[a, b]:.2f});  "
      f"counts / total = ({', '.join(f'{v:.2f}' for v in n_k / n_k.sum())})")

🔍 **What just happened?** A brute-force search over a fine grid lands exactly on the count ratios: $4/7 \approx
0.571$ for the coin and $(0.5, 0.3, 0.2)$ for counts $(5, 3, 2)$. The calculus was right. Now the estimator for a
whole naïve-Bayes model — a few lines of counting. We add an optional pseudo-count `a` that we will need in
Section 8 (with `a = 0` it is the plain MLE).

In [ ]:
# ── 6.2 Fit naive Bayes to the six individuals by counting (slides 39–46) ──
def fit_categorical_nb(X, y, n_values=3, a=0.0):
    '''π = P(Y = 1) and θ[j, i, k] = P(X_i = k | Y = j) by counting (slide 41).

    a = 0 gives the maximum-likelihood estimate; a > 0 adds a imaginary samples of every
    value (the MAP estimate of slide 50, Section 8).
    '''
    X, y = np.asarray(X), np.asarray(y)
    theta = np.zeros((2, X.shape[1], n_values))
    for j in (0, 1):
        Xj = X[y == j]                                         # the samples of class j
        for k in range(n_values):
            theta[j, :, k] = ((Xj == k).sum(axis=0) + a) / (len(Xj) + n_values * a)
    pi = (np.sum(y == 1) + a) / (len(y) + 2 * a)
    return pi, theta


def show_theta(theta, names=("AA (0)", "AT (1)", "TT (2)")):
    '''Print θ as exact fractions, one small table per class.'''
    for j, label in [(1, "Y = 1 (cancer)"), (0, "Y = 0 (healthy)")]:
        tab = pd.DataFrame([[str(Fraction(v).limit_denominator(60)) for v in row] for row in theta[j]],
                           index=[f"P(X{i + 1} = · | {label})" for i in range(theta.shape[1])],
                           columns=list(names)[:theta.shape[2]])
        print(tab.to_string(), "\n")


pi_mle, theta_mle = fit_categorical_nb(X_gen, y_gen)
print(f"π = P(Y = 1) = {Fraction(pi_mle).limit_denominator(60)}   (slide 43: 1/2)\n")
show_theta(theta_mle)
print("slide 45 check: P(X1 = 1 | Y = 1) =", Fraction(theta_mle[1, 0, 1]).limit_denominator(60))

🔍 **What just happened?** Every parameter is a fraction with denominator 3 (three individuals per class) or 6. Check
them by hand against the table of Section 3 — this is exactly what slides 43–46 do on the board:

| | AA (0) | AT (1) | TT (2) | | AA (0) | AT (1) | TT (2) |
|---|---|---|---|---|---|---|---|
| $X_1 \mid Y{=}1$ | 2/3 | 1/3 | **0** | $X_1 \mid Y{=}0$ | **0** | 1/3 | 2/3 |
| $X_2 \mid Y{=}1$ | 2/3 | **0** | 1/3 | $X_2 \mid Y{=}0$ | 1/3 | 1/3 | 1/3 |
| $X_3 \mid Y{=}1$ | 1/3 | 1/3 | 1/3 | $X_3 \mid Y{=}0$ | 1/3 | 1/3 | 1/3 |

The cancer cases (individuals 1–3) have $X_1 \in \{0, 1, 0\}$, hence 2/3, 1/3, 0 — slide 45's $P(X_1 = 1\mid Y=1) =
1/3$. Locus 3 has the same distribution in both classes: on these data it is **useless** for prediction. And three
estimates are exactly **zero** (bold) — remember them; they cause trouble in Section 8.

Altogether the model stores $\pi$ and 18 entries of $\theta$ — 19 numbers — but each row of three sums to 1, so only
$19 - 6 = 13$ are free: exactly the $2\cdot n\cdot 2 + 1 = 13$ of slide 28.

Now classify a new individual with the three steps of slides 30–32.

In [ ]:
# ── 6.3 Classify new individuals (slides 30–32) ──
x_new = (1, 0, 2)                                              # genotypes AT, AA, TT
j0, j1 = nb_joint(x_new, pi_mle, theta_mle)
print(f"new individual x = {x_new}")
print(f"  Step 1: P(Y=1, x) = 1/2 · θ(X1=1|1) · θ(X2=0|1) · θ(X3=2|1) = 1/2 · 1/3 · 2/3 · 1/3 = {j1:.4f}")
print(f"  Step 2: P(Y=0, x) = 1/2 · θ(X1=1|0) · θ(X2=0|0) · θ(X3=2|0) = 1/2 · 1/3 · 1/3 · 1/3 = {j0:.4f}")
print(f"  Step 3: predict Y = {int(j1 > j0)};  posterior P(Y=1 | x) = {j1:.4f} / ({j1:.4f} + {j0:.4f}) = "
      f"{nb_posterior(x_new, pi_mle, theta_mle):.4f}")

post_mle = pd.DataFrame([{"genotype": "".join(map(str, x)), "P(Y=1, x)": nb_joint(x, pi_mle, theta_mle)[1],
                          "P(Y=0, x)": nb_joint(x, pi_mle, theta_mle)[0],
                          "P(Y=1 | x)": nb_posterior(x, pi_mle, theta_mle)} for x in combos])
print(f"\nAll 27 genotypes: posterior exactly 1 for {np.sum(post_mle['P(Y=1 | x)'] == 1)}, exactly 0 for "
      f"{np.sum(post_mle['P(Y=1 | x)'] == 0)}, undefined (0/0) for {post_mle['P(Y=1 | x)'].isna().sum()}:")
print(post_mle.round(4).to_string(index=False, max_rows=30))

🔍 **What just happened?** For $x = (1, 0, 2)$ the joint is $1/27 \approx 0.0370$ for cancer and $1/54 \approx 0.0185$
for healthy, so we predict cancer with posterior probability exactly $2/3$. The classifier now answers for **all 27
genotypes**, including the 21 it has never seen — the payoff of the conditional-independence assumption.

But look at the full table: 18 of the 27 genotypes get a posterior of exactly 1 or exactly 0 — absolute certainty
from six people — and three genotypes ($X_1 = 0$, $X_2 = 1$) give $0/0$: both joints are zero and the model cannot answer at
all. These are the zero estimates at work (Section 8).

Before that, a check against scikit-learn. Its `CategoricalNB` implements the same counting estimator with a
pseudo-count `alpha`; it requires `alpha > 0`, so we use a tiny value to approximate the MLE.

In [ ]:
# ── 6.4 Check against scikit-learn's CategoricalNB ──
sk_nb = CategoricalNB(alpha=1e-10, min_categories=3).fit(X_gen, y_gen)
p_sk = sk_nb.predict_proba(np.array(combos))[:, 1]
p_ours = post_mle["P(Y=1 | x)"].to_numpy()
defined = ~np.isnan(p_ours)
print("largest difference on the 24 genotypes where our MLE posterior is defined:",
      f"{np.abs(p_sk[defined] - p_ours[defined]).max():.1e}")
print("scikit-learn on the 3 'undefined' genotypes:",
      {g: round(float(p), 4) for g, p in zip(post_mle.genotype[~defined], p_sk[~defined])})

🔍 **What just happened?** Where our posterior is defined, scikit-learn agrees to within rounding error. On the three
$0/0$ genotypes, scikit-learn returns 0.6667: with a tiny pseudo-count both joints are tiny but not zero, and their
ratio has a well-defined limit (the tiny pseudo-counts cancel; Section 8 shows this). A number that depends on an
arbitrary smoothing constant should make you uneasy — the real problem is that six individuals cannot support zero
probabilities.

✏️ **Your turn.** Compute by hand the posterior for individual 6's genotype $(1, 2, 1)$, then check with
`nb_posterior((1, 2, 1), pi_mle, theta_mle)`. Is individual 6 classified correctly? Now try all six training
individuals: what is the **training accuracy**, and why is it not a good estimate of how the model will do on new
people?

### 📑 Slide connection — `003_classification-kNN-NB`, slides 34–46

- **What the slides show.** Naïve Bayes is trained by maximising the joint likelihood $\prod_\ell P(Y^\ell, X^\ell)$ (MLE), whereas linear regression maximises the conditional likelihood $\prod_\ell P(Y^\ell\mid X^\ell)$ (MCLE) (slide 34). The log-likelihood splits into a sum of a $\pi$ term and one term per locus (slides 35–38), so each is maximised separately; recalling the coin flip (slide 40) gives the count ratios $\pi = \#(Y=1)/\#\text{samples}$, $\theta_{ijk} = \#(X_i=k, Y=j)/\#(Y=j)$ (slides 39, 41), worked on the genotype table: $P(Y=1)=1/2$, $P(X_1=1\mid Y=1)=1/3$ (slides 42–46).
- **What this notebook adds.** The derivative step for the three-valued die that the slides skip (no Lagrange multipliers needed), a grid check, all 18 conditional probabilities as exact fractions, the posterior for all 27 genotypes, and a comparison with scikit-learn. Note that slide 39 swaps the roles of $i$ and $j$ ($P(X_j=k\mid Y=i)$ with "$Y = 1$" in the formula); slide 41 has the consistent version used here.
- **Exam tip.** Be able to (1) write the log-likelihood, (2) explain why it decomposes, (3) derive $\hat\pi$ by setting a derivative to zero, and (4) fill in a θ table from a small data table by counting — under time pressure.

---
## 7 · A second worked example: two markers → cell type (slide 47)

> 📍 **Where we are.** Slide 47 asks the class to fill in a naïve-Bayes model for a binary class ("Y = 1 iff …")
> from two binary features (eye colour and hair colour). We keep exactly that structure but use a biological
> stand-in that teaches the same lesson.

**Setting (a small SIMULATED illustration).** In single-cell RNA sequencing, a common first task is to label cell
types from marker genes. For ten blood cells we record whether two transcripts were detected:

- $Y = 1$ iff the cell is a **T cell**;
- $X_1 = 1$ iff **CD3E** (a component of the T-cell receptor complex) was detected;
- $X_2 = 1$ iff **IL7R** (the interleukin-7 receptor, expressed by many T cells but also by some other immune cells)
  was detected.

Single-cell data have many **dropouts** (a transcript that is present but not captured), so a T cell can show
$X_1 = 0$. Fill in the ten blanks of slide 47 by counting.

In [ ]:
# ── 7.1 Fill in the blanks of slide 47 for the T-cell stand-in ──
cells_df = pd.DataFrame({"CD3E detected (X1)": [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
                         "IL7R detected (X2)": [1, 0, 1, 1, 0, 1, 0, 0, 1, 0],
                         "T cell (Y)":         [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]},
                        index=[f"cell {i}" for i in range(1, 11)])
print(cells_df.to_string(), "\n")
Xc, yc = cells_df.iloc[:, :2].to_numpy(), cells_df.iloc[:, 2].to_numpy()
pi_c, theta_c = fit_categorical_nb(Xc, yc, n_values=2)
for j in (1, 0):
    print(f"P(Y={j}) = {pi_c if j else 1 - pi_c:.2f}")
    for i in range(2):
        for k in (1, 0):
            print(f"   P(X{i + 1}={k} | Y={j}) = {Fraction(theta_c[j, i, k]).limit_denominator(12)}")
print("\nPosterior P(T cell | markers):")
for x in itertools.product([0, 1], repeat=2):
    print(f"   CD3E={x[0]}, IL7R={x[1]}:  {nb_posterior(x, pi_c, theta_c):.4f}")

🔍 **What just happened?** Counting gives $P(Y{=}1) = 0.4$, $P(X_1{=}1\mid Y{=}1) = 3/4$, $P(X_2{=}1\mid Y{=}1) = 3/4$,
$P(X_2{=}1 \mid Y{=}0) = 1/3$ — and $P(X_1{=}1\mid Y{=}0) = 0$, because none of the six non-T cells happened to show CD3E.
As a result, **every** cell with CD3E detected is declared a T cell with probability exactly 1, whatever else we
know about it. Yet CD3E reads do turn up outside T cells — at low levels in some other immune cells, and in
"doublets" (two cells captured together by accident): in a larger data set we would surely meet a non-T cell with
$X_1 = 1$, and the model would call it a T cell with total certainty.
A cell with neither marker gets $P = 0.0588$; a cell with only IL7R gets $0.2727$.

This is word for word the situation of slide 48: *"nobody in our class has $X_1 = 1$ and $Y = 0$, then
$P(X_1 = 1 \mid Y = 0) = 0$."*

---
## 8 · Subtlety #1: zero counts, and the MAP fix (slides 48–50)

> 📍 **Where we are.** Both worked examples produced zero estimates. One zero factor makes the whole product
> $P(Y)\prod_i P(X_i \mid Y)$ zero, however strong the evidence from all other features (slide 48: "Why worry about
> just one parameter out of many?"). With thousands of features, zeros are almost guaranteed.

**The fix: MAP estimation** (slide 49, Module 05). Instead of the $\theta$ that maximises $P(\mathcal D\mid\theta)$, choose
the $\theta$ that maximises the posterior

$$\hat\theta_{MAP} \;=\; \arg\max_\theta P(\theta \mid \mathcal D) \;=\; \arg\max_\theta \frac{P(\mathcal D\mid\theta)\,P(\theta)}{P(\mathcal D)} .$$

*In words:* weigh how well $\theta$ explains the data against how plausible $\theta$ was before seeing the data.

🧮 **Math refresher — Beta and Dirichlet priors as imaginary data.** For a probability vector
$(\theta_0, \theta_1, \theta_2)$ the conjugate prior is the **Dirichlet** distribution,
$P(\theta) \propto \theta_0^{\alpha_0-1}\theta_1^{\alpha_1-1}\theta_2^{\alpha_2-1}$ (for two outcomes it is called the
**Beta** distribution). Multiplying by the likelihood $\theta_0^{n_0}\theta_1^{n_1}\theta_2^{n_2}$ simply *adds the
exponents*: the posterior is $\propto \prod_k \theta_k^{\,n_k + \alpha_k - 1}$ — it looks exactly like a likelihood
with $\alpha_k - 1$ extra observations of value $k$. Maximising it repeats the counting derivation of Section 6 with
the counts increased. Writing $a = \alpha_k - 1$ for the number of **imaginary samples** of each value:

$$\hat\theta_{ijk}^{MAP} \;=\; \frac{\#(X_i = k,\, Y = j) + a}{\#(Y = j) + 3a}, \qquad \hat\pi^{MAP} = \frac{\#(Y=1) + a}{\#\text{samples} + 2a} .$$

*In words:* pretend that, before collecting data, we had already seen every genotype $a$ times in every class. No
estimate can then be zero. $a = 1$ is called **Laplace smoothing** ("add-one"). Slide 50 writes the same idea with
separate pseudo-counts $\alpha, \beta$ ("Only difference: 'imaginary' examples"). ⚠️ Two details of slide 50: its
$\theta$ formula is printed for $Y = 1$ (it holds for any class $j$) and with a two-outcome (Beta) denominator; for a
three-valued genotype the denominator adds the imaginary counts of **all three** values, hence the $3a$ above.

In [ ]:
# ── 8.1 MAP estimates with one imaginary sample per genotype (Laplace smoothing, a = 1) ──
pi_map, theta_map = fit_categorical_nb(X_gen, y_gen, a=1.0)
print(f"π (MAP) = {Fraction(pi_map).limit_denominator(60)}\n")
show_theta(theta_map)
post_map = np.array([nb_posterior(x, pi_map, theta_map) for x in combos])
print(f"posterior exactly 0 or 1: {np.sum((post_map == 0) | (post_map == 1))};  undefined: {np.isnan(post_map).sum()};"
      f"  range of P(Y=1|x): {post_map.min():.3f} – {post_map.max():.3f}")
for x in [(1, 0, 2), (2, 0, 0), (0, 1, 0)]:
    i = combos.index(x)
    print(f"genotype {x}: MLE posterior {post_mle['P(Y=1 | x)'][i]:.4f}  →  MAP posterior {post_map[i]:.4f}")

🔍 **What just happened?** With one imaginary sample of each genotype per class, every $\theta$ is now
$(\text{count} + 1)/6$ — no zeros. No genotype is classified with absolute certainty any more and none is
undefined; posteriors range from 0.14 to 0.82. The individual with $(2, 0, 0)$, previously "healthy with
probability 1" because nobody in the cancer group had TT at locus 1, now gets $P(\text{cancer}) = 1/3$: the other loci
are allowed to speak. Smoothing also pulls confident estimates towards the middle: $(1, 0, 2)$ goes from $2/3$ to
$0.6$.

How much smoothing? $a$ is a knob: small $a$ trusts the data, large $a$ trusts the prior (all genotypes equally
likely, which makes the loci uninformative).

In [ ]:
# ── 8.2 How the pseudo-count size changes the posterior ──
a_grid = np.logspace(-3, 3, 61)
track = {(0, 2, 0): "tab:blue", (2, 0, 0): "tab:red", (0, 1, 0): "tab:green"}
fig, ax = plt.subplots(figsize=(6, 3.4))
for x, col in track.items():
    p = [nb_posterior(x, *fit_categorical_nb(X_gen, y_gen, a=a)) for a in a_grid]
    ax.plot(a_grid, p, color=col, lw=2, label=f"genotype {x}")
    print(f"{x}: a = 0.001 → {p[0]:.3f},  a = 1 → {p[30]:.3f},  a = 1000 → {p[-1]:.3f}")
ax.axhline(0.5, color="gray", ls=":", lw=1); ax.axvline(1, color="gray", ls="--", lw=1)
ax.text(1.15, 0.92, "Laplace (a = 1)", fontsize=7, color="gray")
ax.set_xscale("log"); ax.set_ylim(-0.03, 1.03)
ax.set_xlabel("pseudo-count a (imaginary samples per genotype, log scale)"); ax.set_ylabel("P(Y = 1 | x)")
ax.set_title("smoothing moves every posterior towards the prior"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** For tiny $a$ the posteriors approach the MLE: essentially 1 for individual 3's genotype
$(0, 2, 0)$, essentially 0 for $(2, 0, 0)$, and 0.667 for the "undefined" genotype $(0, 1, 0)$ — the limit
scikit-learn reported in 6.4 (both classes contain one zero factor, and the two tiny pseudo-counts cancel in the
ratio: check by hand that the ratio of the two joints is $(2 + a)/(1 + a)$). For huge $a$ the imaginary
samples swamp the six real ones, every $\theta$ tends to 1/3 and every posterior to the prior 0.5. Useful values lie
in between.

⚠️ **Choosing $a$** is a model-selection decision (validation data or cross-validation, Module 23). With large
training sets it matters little: one imaginary sample changes a count of 1,000 by 0.1 %, but a count of 3 by 33 %.

In [ ]:
# ── 8.3 scikit-learn's alpha is exactly our pseudo-count a ──
for a in (1.0, 0.5):
    sk = CategoricalNB(alpha=a, min_categories=3).fit(X_gen, y_gen)
    ours = np.array([nb_posterior(x, *fit_categorical_nb(X_gen, y_gen, a=a)) for x in combos])
    print(f"a = alpha = {a}: largest difference over the 27 genotypes = "
          f"{np.abs(sk.predict_proba(np.array(combos))[:, 1] - ours).max():.1e}")

🔍 **What just happened?** The two implementations agree to machine precision. (scikit-learn estimates the class
prior without smoothing by default; here it makes no difference because both classes have three individuals, so
$\hat\pi = 1/2$ either way.)

---
## 9 · Subtlety #2: when features are not conditionally independent (slide 51)

> 📍 **Where we are.** The naïve-Bayes assumption is almost never exactly true. Slide 51 asks: what does a violation
> do to $P(Y \mid X)$? Its special case: *what if we add two copies, $X_i = X_k$?*

🧮 **The log-odds view.** Divide the two numerators of the naïve-Bayes classifier and take logs:

$$\log\frac{P(Y{=}1\mid x)}{P(Y{=}0\mid x)} \;=\; \log\frac{\pi}{1-\pi} \;+\; \sum_{i=1}^{n} \log\frac{\theta_{i1x_i}}{\theta_{i0x_i}} .$$

*In words:* naïve Bayes starts from the prior log-odds and **adds one "evidence" term per feature** — the log of how
much more likely that genotype is among cases than among controls. The posterior is recovered with the logistic
function, $P(Y{=}1 \mid x) = 1/(1 + e^{-\text{log-odds}})$ — keep this in mind for [Module 09](09_logistic_regression.ipynb).

A duplicated feature adds its evidence **twice**; ten copies add it eleven times. Nothing new has been learned, but
the model becomes ever more certain.

In [ ]:
# ── 9.1 Duplicating a locus in the genotype example (MAP model, a = 1) ──
x_base = np.array([1, 0, 2])
print(" copies of X2 | log-odds | P(Y=1 | x)")
for m in [0, 1, 2, 5, 10]:
    X_dup = np.c_[X_gen, np.repeat(X_gen[:, [1]], m, axis=1)]       # X2 copied m times
    pi_d, th_d = fit_categorical_nb(X_dup, y_gen, a=1.0)
    x_dup = np.r_[x_base, np.repeat(x_base[1], m)]
    p = nb_posterior(x_dup, pi_d, th_d)
    print(f"      {m:2d}      |  {np.log(p / (1 - p)):+.3f}  |  {p:.3f}")
print(f"\nevidence of X2 = 0: log[θ(X2=0|Y=1) / θ(X2=0|Y=0)] = log[(3/6)/(2/6)] = {np.log(1.5):.3f} per copy")

🔍 **What just happened?** Without copies the MAP posterior of $(1, 0, 2)$ is 0.6 (log-odds 0.405). Each copy of
locus 2 adds the same evidence, $\log 1.5 = 0.405$, so after ten copies the log-odds is $11 \times 0.405 = 4.46$
and the classifier is 98.9 % sure — on the basis of six people. The *decision* (cancer) never changed; only the
confidence did.

**Biological version: linkage disequilibrium.** Neighbouring SNPs on a chromosome are inherited together, so a
causal SNP often has many near-copies on a genotyping chip. Let us simulate a case/control cohort (SIMULATED) with
five independent risk SNPs plus $m$ "LD partners" of SNP 1 — each partner copies SNP 1's two alleles but re-draws each
allele with probability 5 % (correlation ≈ 0.95). We train naïve Bayes (a = 1) on 200 individuals and test on 5,000.

🧮 **Measuring probability quality: the log-loss.** Accuracy only checks the decision. To judge the
*probabilities*, use the average negative log-probability given to the true class,

$$\text{log-loss} \;=\; -\frac{1}{N}\sum_{\ell=1}^{N}\Big[y^\ell \log \hat p^\ell + (1-y^\ell)\log(1-\hat p^\ell)\Big],\qquad \hat p^\ell = \hat P(Y{=}1\mid x^\ell).$$

*In words:* each test individual "pays" $-\log$ of the probability the model gave to what actually happened —
little for a confident correct answer, a lot for a confident wrong one ($-\log 0.01 = 4.6$). Always guessing 0.5
costs $\log 2 = 0.693$. Lower is better. This is the negative of the (conditional) log-likelihood of the test data,
the quantity logistic regression maximises in Module 09.

In [ ]:
# ── 9.2 SNPs in linkage disequilibrium: accuracy vs probability quality (SIMULATED) ──
rng = np.random.default_rng(SEED + 9)
P_CASE, P_CTRL = np.array([0.50, 0.45, 0.45, 0.45, 0.45]), np.full(5, 0.30)   # risk-allele frequencies


def simulate_cohort(n, rng, max_copies=20, redraw=0.05):
    '''SIMULATED case/control cohort: 5 independent risk SNPs + max_copies LD partners of SNP 1.'''
    y = rng.integers(0, 2, n)
    G = rng.binomial(2, np.where(y[:, None] == 1, P_CASE, P_CTRL))
    alleles = np.c_[G[:, 0] >= 1, G[:, 0] >= 2].astype(int)           # SNP 1 as two 0/1 alleles
    partners = []
    for _ in range(max_copies):
        swap = rng.random((n, 2)) < redraw                              # re-draw each allele with prob. 5 %
        partners.append(np.where(swap, rng.binomial(1, 0.4, (n, 2)), alleles).sum(axis=1))
    return np.c_[G, np.array(partners).T], y


def nb_predict_proba(X, pi, theta):
    '''P(Y = 1 | x) for many rows: prior log-odds + summed evidence, through the logistic function.'''
    idx = np.arange(X.shape[1])
    log_odds = np.log(pi / (1 - pi)) + np.sum(np.log(theta[1, idx, X] / theta[0, idx, X]), axis=1)
    return sigmoid(log_odds)


copies_list, R = [0, 1, 2, 5, 10, 20], 20
res = {m: {"acc": [], "logloss": [], "conf": [], "same": []} for m in copies_list}
for r in range(R):
    G_tr, yc_tr = simulate_cohort(200, rng)
    G_te, yc_te = simulate_cohort(5000, rng)
    for m in copies_list:
        cols = np.arange(5 + m)                                         # 5 risk SNPs + the first m partners
        p = np.clip(nb_predict_proba(G_te[:, cols], *fit_categorical_nb(G_tr[:, cols], yc_tr, a=1.0)), 1e-12, 1 - 1e-12)
        pred = (p > 0.5).astype(int)
        base = pred if m == 0 else base
        res[m]["acc"].append(accuracy(yc_te, pred)); res[m]["same"].append(np.mean(pred == base))
        res[m]["logloss"].append(-np.mean(yc_te * np.log(p) + (1 - yc_te) * np.log(1 - p)))
        res[m]["conf"].append(np.mean(np.maximum(p, 1 - p)))
print(f"correlation of SNP 1 with one LD partner: {np.corrcoef(G_te[:, 0], G_te[:, 5])[0, 1]:.2f}\n")
print("partners | test accuracy | decisions same as m=0 | log-loss | mean confidence")
for m in copies_list:
    s = {k: np.mean(v) for k, v in res[m].items()}
    print(f"   {m:2d}    |     {s['acc']:.3f}     |         {s['same']:.3f}         |  {s['logloss']:.3f}  |     {s['conf']:.3f}")

In [ ]:
# ── 9.3 Plot: the decision degrades slowly, the probabilities fall apart ──
summ = {k: np.array([np.mean(res[m][k]) for m in copies_list]) for k in ["acc", "same", "logloss", "conf"]}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.4))
a1.plot(copies_list, summ["acc"], "o-", color="tab:blue", label="test accuracy")
a1.plot(copies_list, summ["same"], "s--", color="tab:green", label="decisions unchanged vs no partners")
a1.set_ylim(0.5, 1.02); a1.set_xlabel("number of LD partners of SNP 1"); a1.set_ylabel("fraction")
a1.set_title("classification: modest damage"); a1.legend(fontsize=7)
a2.plot(copies_list, summ["logloss"], "o-", color="tab:red", label="log-loss (lower = better)")
a2.axhline(np.log(2), color="gray", ls=":", lw=1); a2.text(12, np.log(2) - 0.22, "always say 0.5", fontsize=7, color="gray")
a2.set_xlabel("number of LD partners of SNP 1"); a2.set_ylabel("test log-loss")
a2b = a2.twinx(); a2b.plot(copies_list, summ["conf"], "^--", color="tab:purple", label="mean confidence")
a2b.set_ylabel("mean max(p, 1 − p)", color="tab:purple"); a2b.spines["right"].set_visible(True)
a2.set_title("probabilities: over-confident and badly calibrated"); a2.legend(fontsize=7, loc="upper left")
plt.suptitle("naïve Bayes with correlated SNPs (SIMULATED, 20 replicate cohorts)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** As LD partners are added, test accuracy drifts down only gently (from about 0.69 to
0.63 with 20 partners), and most individual decisions stay the same as without partners. But the probabilities
deteriorate dramatically: mean confidence climbs from about 0.71 to 0.96, and the log-loss rises from 0.59 — better
than guessing — to about 3, far *worse* than always saying 0.5. The model has become very sure and often wrong.

This is slide 51's message, made quantitative: naïve Bayes "often [gives] the right classification, even when not
the right probability" (Domingos & Pazzani). The decision only needs the log-odds to have the right **sign**;
double-counted evidence inflates its **size**. If you need trustworthy probabilities (risk prediction, ranking
patients for follow-up), naïve Bayes with correlated features is dangerous. Practical remedies: remove redundant
features (LD pruning in genetics), model the dependent pair jointly (slide 54), or use a discriminative model such as
logistic regression, which shares the weight among correlated features instead of counting them twice (Module 09).

✏️ **Your turn.** Set `redraw=0.5` in `simulate_cohort` (partners only weakly correlated with SNP 1) and re-run 9.2.
Does the log-loss still explode? What does that tell you about *how much* dependence matters?

### 📑 Slide connection — `003_classification-kNN-NB`, slides 47–51

- **What the slides show.** Slide 47: fill in $P(Y)$ and $P(X_1\mid Y)$, $P(X_2\mid Y)$ for a binary class from two binary features (hair/eye colour). Subtlety #1 (slide 48): an MLE can be zero ("nobody in our class has $X_1 = 1$ and $Y = 0$"), which zeroes the whole product. Slides 49–50: MAP estimation with Beta/Dirichlet priors = MLE plus "imaginary" examples. Subtlety #2 (slide 51): the $X_i$ are often not conditionally independent; NB often still gives the right class but not the right probability (Domingos & Pazzani); special case $X_i = X_k$.
- **What this notebook adds.** A biological stand-in for slide 47 that produces exactly slide 48's zero; the Dirichlet derivation of the pseudo-count formula (and the $3a$ denominator for three genotypes); the effect of the pseudo-count size; a check against scikit-learn's `alpha`; the log-odds view of double counting; and an LD simulation that separates accuracy (barely hurt) from log-loss (ruined).
- **Exam tip.** Know the MAP formula with pseudo-counts and what happens as $a\to 0$ (MLE) and $a\to\infty$ (uniform, posterior → prior). For duplicated features, be able to show that the log-odds evidence of the copied feature is counted twice. (The slide cites Domingos & Pazzani 1996; the journal version is 1997.)

---
## 10 · Continuous features: Gaussian naïve Bayes on real leukaemia data (slides 52–55)

> 📍 **Where we are.** Genotypes take three values, but most biological measurements are continuous: expression
> levels, protein abundances, pixel intensities (slide 52 classifies brain images: schizophrenic vs normal, which
> task, which word). Naïve Bayes needs $P(X_i \mid Y = y_k)$ for a real-valued $X_i$.

**The common approach** (slide 53): assume each feature is **Gaussian within each class**, with its own mean and
variance:

$$P(X_i = x \mid Y = k) \;=\; \mathcal N(x;\,\mu_{ik},\,\sigma_{ik}^2) \;=\; \frac{1}{\sqrt{2\pi\sigma_{ik}^2}}\exp\!\Big(-\frac{(x-\mu_{ik})^2}{2\sigma_{ik}^2}\Big).$$

*In words:* among, say, AML patients, the expression of gene $i$ scatters in a bell curve around a typical value
$\mu_{ik}$ with spread $\sigma_{ik}$ (Module 02). The model for $Y \mid X$ is otherwise unchanged:
$P(Y = k \mid x) \propto P(Y = k)\prod_i \mathcal N(x_i;\mu_{ik},\sigma^2_{ik})$.

**MLE is again closed-form** (Module 05 derived it for one Gaussian): $\hat\mu_{ik}$ is the mean of feature $i$ over
the training samples of class $k$, and $\hat\sigma^2_{ik}$ is their average squared deviation from that mean (dividing
by $N_k$, not $N_k - 1$). Counting has become averaging — still no iterations.

⚠️ **Computing with many features.** A product of thousands of densities underflows to 0.0 on a computer. We work
with logs — $\log\prod = \sum\log$ — and normalise with the toolkit's `logsumexp`, which computes
$\log\sum_k e^{a_k}$ without ever forming the tiny numbers $e^{a_k}$.

In [ ]:
# ── 10.1 Gaussian naive Bayes from scratch ──
def fit_gnb(X, y, var_add=0.0):
    '''Maximum-likelihood Gaussian naive Bayes: class priors, per-class means and variances (divide by N_k).

    var_add is added to every variance — a 'variance floor' (Section 10.2); 0 gives the pure MLE.
    '''
    classes = np.unique(y)
    prior = np.array([np.mean(y == c) for c in classes])
    mu = np.array([X[y == c].mean(axis=0) for c in classes])
    var = np.array([X[y == c].var(axis=0) for c in classes]) + var_add
    return prior, mu, var


def gnb_log_joint(X, prior, mu, var):
    '''log P(Y = c) + Σ_i log N(x_i; μ_ci, σ²_ci) for every sample (rows) and class (columns).'''
    with np.errstate(divide="ignore", invalid="ignore"):          # zero variances are diagnosed in 10.2
        return np.stack([np.log(prior[c]) - 0.5 * np.sum(np.log(2 * np.pi * var[c]) + (X - mu[c]) ** 2 / var[c],
                                                          axis=1) for c in range(len(prior))], axis=1)


def gnb_predict_proba(X, prior, mu, var):
    '''Posterior P(Y = c | x), normalised in log space with logsumexp (no underflow).'''
    lj = gnb_log_joint(X, prior, mu, var)
    return np.exp(lj - logsumexp(lj, axis=1)[:, None])

**Choosing genes — on the training samples only.** With 3,571 genes and 38 training patients we start with a few
genes that differ most between ALL and AML. Golub et al. ranked genes by a **signal-to-noise** score,

$$s_i \;=\; \frac{\mu_{i,\text{AML}} - \mu_{i,\text{ALL}}}{\sigma_{i,\text{AML}} + \sigma_{i,\text{ALL}}},$$

*In words:* the difference between the class means, measured in units of the within-class spread; large positive =
higher in AML, large negative = higher in ALL.

⚠️ **Selection bias.** The score must be computed on the **38 training patients only**. Ranking genes with all 72
patients lets the test patients help choose the features, and the test accuracy becomes optimistic (Ambroise &
McLachlan 2002; Module 23 measures how much).

In [ ]:
# ── 10.2 Rank genes on the TRAINING patients and fit a 2-gene Gaussian naive Bayes ──
Xg_tr, yg_tr, Xg_te, yg_te = Xg[is_tr], yg[is_tr], Xg[~is_tr], yg[~is_tr]


def signal_to_noise(X, y):
    '''Golub et al.'s score (μ_AML − μ_ALL) / (σ_AML + σ_ALL) for every column of X.'''
    a, b = X[y == 1], X[y == 0]
    with np.errstate(divide="ignore", invalid="ignore"):          # genes constant in both classes → 0
        return np.nan_to_num((a.mean(axis=0) - b.mean(axis=0)) / (a.std(axis=0) + b.std(axis=0)))


s2n = signal_to_noise(Xg_tr, yg_tr)                               # 38 training patients only!
rank = np.argsort(-np.abs(s2n))
KNOWN = {"M27891_at": "cystatin C (CST3)", "X95735_at": "zyxin (ZYX)", "M23197_at": "CD33",
         "U50136_rna1_at": "leukotriene C4 synthase (LTC4S)",
         "U22376_cds2_s_at": "MYB", "M84526_at": "complement factor D / adipsin (CFD)"}
for r, j in enumerate(rank[:8], 1):
    print(f"{r}. {golub['genes'][j]:18s} s = {s2n[j]:+.2f}  higher in {'AML' if s2n[j] > 0 else 'ALL'}  "
          f"{KNOWN.get(golub['genes'][j], '')}")

g2 = rank[:2]
prior2, mu2, var2 = fit_gnb(Xg_tr[:, g2], yg_tr)
print(f"\nparameters: priors {np.round(prior2, 3)} (ALL, AML); 2 genes × 2 classes × (mean, variance) + 1 = 9 free")
print(pd.DataFrame({"mean ALL": mu2[0], "sd ALL": np.sqrt(var2[0]), "mean AML": mu2[1], "sd AML": np.sqrt(var2[1])},
                   index=golub["genes"][g2]).round(3).to_string())
post2 = gnb_predict_proba(Xg_te[:, g2], prior2, mu2, var2)[:, 1]
pred2 = (post2 > 0.5).astype(int)
sk2 = GaussianNB(var_smoothing=0.0).fit(Xg_tr[:, g2], yg_tr)
print(f"\ntest accuracy: {accuracy(yg_te, pred2):.3f} ({np.sum(pred2 != yg_te)} errors / {len(yg_te)});  "
      f"scikit-learn GaussianNB agrees on all test patients: {np.array_equal(sk2.predict(Xg_te[:, g2]), pred2)}")
for i in np.flatnonzero(pred2 != yg_te):
    print(f"   misclassified test patient {golub['sample_id'][~is_tr][i]}: truly {'AML' if yg_te[i] else 'ALL'}, "
          f"P(AML | x) = {post2[i]:.3f}")

🔍 **What just happened?** The top-ranked genes, chosen with training patients only, make biological sense:
**cystatin C**, **zyxin**, **leukotriene C4 synthase** (the gene Module 05 picked) and the myeloid surface marker
**CD33** are higher in AML (myeloid leukaemia), and the
transcription factor **MYB** (rank 7) is higher in ALL. (Gene names are the original Affymetrix probe annotations; verify in a
database before building on them.) Two genes, nine parameters, and Gaussian naïve Bayes classifies 31 of the 34
independent test patients correctly (0.912 vs the 0.588 baseline of Section 1). Two AML patients are called ALL, one
ALL patient is called AML with $P(\text{AML}) = 0.999$ — an over-confident mistake. Our implementation and
scikit-learn's agree.

In [ ]:
# ── 10.3 The decision boundary of Gaussian naive Bayes is curved ──
fig, ax = plt.subplots(figsize=(5.4, 4.2))
plot_decision_regions(lambda Q: gnb_predict_proba(Q, prior2, mu2, var2).argmax(axis=1), Xg_tr[:, g2], yg_tr, ax=ax,
                      resolution=200, class_names=["ALL (train)", "AML (train)"], pad=0.3)
for c, name in [(0, "ALL (test)"), (1, "AML (test)")]:
    m = yg_te == c
    ax.scatter(*Xg_te[m][:, g2].T, s=40, facecolor="none", edgecolor=CLASS_COLORS[c], lw=1.3, marker="D", label=name)
wrong = pred2 != yg_te
ax.scatter(*Xg_te[wrong][:, g2].T, s=140, facecolor="none", edgecolor="k", lw=1, label="misclassified test")
ax.set_xlabel(f"{golub['genes'][g2[0]]} (log10 expression)"); ax.set_ylabel(f"{golub['genes'][g2[1]]} (log10 expression)")
ax.set_title(f"Gaussian naïve Bayes, 2 genes ({golub['source'].split('(')[0].strip()})")
ax.legend(fontsize=7, loc="lower right"); plt.tight_layout(); plt.show()

🔍 **What just happened?** The boundary is **curved**, not straight. Because each class has its own variances —
AML samples are much more tightly spread along the vertical (zyxin) gene than ALL samples — the log-odds contains
squared terms $x_i^2$ that do not cancel, and the boundary is a quadratic curve (a piece of an ellipse or hyperbola). (If both classes shared the same
variances, the quadratic terms would cancel and the boundary would be a straight line — Module 09 shows that this
special case gives exactly the logistic-regression form.) The test patients (diamonds) fall mostly on the right side;
the circled ones are the three errors (one AML patient sits at the top left, with cystatin C close to the detection
floor).

### 10.1 More genes — and a continuous version of the zero-count problem

How many parameters does Gaussian naïve Bayes need (slide 54's question)? For $k$ classes and $n$ features: a mean
and a variance per feature per class, plus $k - 1$ free class probabilities:

$$\#\theta_{\text{GNB}} \;=\; 2\,n\,k + (k - 1) .$$

*In words:* still linear in the number of features. For two classes this is $4n + 1$ (slide 54: "n features → 4n
parameters" for the class-conditional part, plus $P(Y)$). With all 3,571 genes: 14,285 parameters — large, but each one
is estimated from 11 or 27 patients. A full multivariate Gaussian with a covariance matrix per class would need
$n(n+1)/2 \approx 6.4$ million numbers per class: hopeless with 38 patients.

Let us use more genes and compare with k-NN on the same (standardised) genes.

In [ ]:
# ── 10.4 More genes: naive Bayes with and without a variance floor, and k-NN ──
rows = []
for G in [2, 10, 20, 50, 100]:
    g = rank[:G]
    n_zero = int(np.sum((Xg_tr[yg_tr == 1][:, g].var(axis=0) == 0) | (Xg_tr[yg_tr == 0][:, g].var(axis=0) == 0)))
    err = {}
    for label, va in [("GNB, MLE (var + 1e-6)", 1e-6), ("GNB, var floor 0.01", 0.01)]:
        p = gnb_predict_proba(Xg_te[:, g], *fit_gnb(Xg_tr[:, g], yg_tr, var_add=va))[:, 1]
        err[label] = int(np.sum((p > 0.5) != yg_te))
    Zg_tr, m_g, s_g = standardize(Xg_tr[:, g]); Zg_te, _, _ = standardize(Xg_te[:, g], m_g, s_g)
    err["3-NN (standardised)"] = int(np.sum(knn_predict(Zg_tr, yg_tr, Zg_te, 3) != yg_te))
    rows.append({"genes": G, "genes with zero within-class variance": n_zero, **err})
print("test errors out of 34 (genes ranked on the 38 training patients):")
print(pd.DataFrame(rows).to_string(index=False))

🔍 **What just happened?** With 10–20 genes all methods make 1–3 errors. But at 50 and 100 genes plain Gaussian
naïve Bayes suddenly makes 8 errors, while the same model with a small **variance floor** (0.01 added to every
variance, i.e. a minimum standard deviation of 0.1 log10 units) and 3-NN make a single error. The column in the middle gives the
reason: some selected genes have **zero variance within a class** in the training data. The next cell looks at one.

In [ ]:
# ── 10.5 The culprit: a gene sitting at the detection floor in every AML training patient ──
g50 = rank[:50]
culprit = g50[np.argmin(Xg_tr[yg_tr == 1][:, g50].var(axis=0))]
print(f"gene {golub['genes'][culprit]} (rank {list(rank).index(culprit) + 1} by signal-to-noise)")
for name, X_, y_ in [("train", Xg_tr, yg_tr), ("test", Xg_te, yg_te)]:
    for c, lab in [(1, "AML"), (0, "ALL")]:
        print(f"  {name} {lab}: {np.round(np.sort(X_[y_ == c, culprit]), 2)}")
print(f"\nfraction of all expression values in the data set equal to the floor value 2.0: {np.mean(Xg == 2.0):.2f}")

🔍 **What just happened?** In all eleven AML training patients this gene reads exactly **2.0** — the detection floor
of the preprocessing (expression values below 100 were set to 100, and $\log_{10} 100 = 2$; about a fifth of all
measurements in the data set sit there). Its estimated AML variance is therefore **zero**: the fitted AML "bell curve"
is an infinitely thin spike at 2.0. Any test patient with a value even slightly above 2.0 — like the AML test patients
reading 2.44 and 2.49 — is then *infinitely* unlikely to be AML, whatever the other 49 genes say, and an ALL patient at
the floor looks infinitely AML-like. (Mathematically the density divides by $\sigma^2 = 0$; we added a microscopic
$10^{-6}$ only so that the computer can evaluate it — any tiny value gives the same answer.)

This is **subtlety #1 in continuous form**: an estimate that is exactly zero (here a variance, there a probability)
because the training sample is small, which then vetoes all other evidence. The cure is the same idea as
pseudo-counts — shrink the estimate away from the impossible value with a little prior information (a variance
floor; scikit-learn's `var_smoothing` does this relative to the largest variance).

⚠️ **We printed several gene numbers only to illustrate this failure.** Picking the number of genes (or the floor)
by looking at the test errors would be cheating; in a real analysis both are chosen with cross-validation on the
training patients (Module 23).

✏️ **Your turn.** Re-run 10.4 with `var_add=0.001` and `0.1` for the floored model. Which floors work? Then rank the
genes by the *absolute difference of class means* instead of the signal-to-noise score: do zero-variance genes enter
the top 50 sooner or later, and why?

### 10.2 The other two questions of slide 54

- **Mixed discrete and continuous features?** Yes. The likelihood is a product over features, and each factor can
  have its own distribution: $P(Y \mid x) \propto P(Y)\prod_{i \in \text{genotypes}} \theta_{iYx_i}\prod_{i \in
  \text{expression}}\mathcal N(x_i;\mu_{iY},\sigma^2_{iY})$. A model combining SNPs and expression is just naïve Bayes
  with both kinds of factor.
- **Modelling just 2 of the n attributes as dependent?** Replace their two factors by one joint factor
  $P(X_i, X_j \mid Y)$. For two genotype loci that is a 9-cell table per class (8 free parameters instead of 2 + 2),
  for two Gaussian genes a 2-D Gaussian with a covariance term. Allowing a few chosen dependencies is the idea behind
  Bayesian networks (Modules 15–17).

### 📑 Slide connection — `003_classification-kNN-NB`, slides 52–55

- **What the slides show.** Continuous $X_i$ such as image pixels (fMRI: schizophrenic vs normal brain, which task, which word, slide 52); the common approach assumes $P(X_i\mid Y=y_k)$ is Gaussian (slide 53); questions to think about: mixed discrete/continuous features, modelling 2 of n attributes as dependent via $P(X_i, X_j\mid Y)$, and the parameter count for k classes ($P(Y)$: $k-1$; per feature 2 per class; "n features → 4n parameters" for two classes) (slide 54). Summary (slide 55): NB models $P(X\mid Y)$ and $P(Y)$, closed-form MLE, no iterative learning — in contrast with logistic regression.
- **What this notebook adds.** A from-scratch Gaussian NB computed in log space, applied to the real Golub data with the original train/test split and genes selected on training patients only; the curved (quadratic) decision boundary; the full answer $2nk + (k-1)$; a comparison with k-NN; and a real-data failure — zero within-class variance from a detection floor — that mirrors the zero-count subtlety.
- **Exam tip.** Answer slide 54's count as $2nk$ (means and variances) $+\,(k-1)$ (priors). Know why the Gaussian NB boundary is quadratic with class-specific variances and linear with shared variances.

---
## 📌 Key takeaways

- **Classification** learns $f: X \to Y$ for discrete $Y$; quality is judged on **test** data never used in training,
  against a baseline such as "always predict the majority class".
- **k-NN** (non-parametric, no learning): majority vote of the K nearest training samples. Small K = wiggly, noisy
  boundaries (1-NN has zero training error); large K = over-smoothing; K = N = one class for everyone. Distances need
  comparable feature scales; the cost is paid at prediction time and grows with N.
- **Naïve Bayes** (parametric, generative) models $P(Y)$ and $P(X \mid Y)$ and obtains $P(Y \mid X)$ with Bayes' rule.
  Its **conditional-independence assumption** $P(X_1\ldots X_n\mid Y)=\prod_i P(X_i\mid Y)$ cuts the parameter count from
  $3^n$ (or $(3^n-1)\times 2+1$) to $2\cdot n\cdot 2+1$.
- **MLE = counting:** the log-likelihood decomposes, so $\hat\pi = \#(Y{=}1)/L$ and
  $\hat\theta_{ijk} = \#(X_i{=}k, Y{=}j)/\#(Y{=}j)$ — closed form, no iterations. On the slides' table:
  $\pi = 1/2$, $P(X_1{=}1 \mid Y{=}1) = 1/3$, and $P(Y{=}1\mid 1,0,2) = 2/3$.
- **Subtlety #1 — zero counts** veto all other evidence; **MAP** with Beta/Dirichlet priors adds imaginary samples,
  $(\text{count} + a)/(N_j + 3a)$. Its continuous twin is a zero within-class variance (cured by a variance floor).
- **Subtlety #2 — dependent features** are double-counted in the log-odds: decisions often survive, probabilities
  become over-confident (log-loss explodes).
- **Gaussian naïve Bayes** handles continuous features with per-class means and variances ($2nk + k - 1$ parameters)
  and gives a quadratic boundary; on real leukaemia data two training-selected genes already classify 31/34 test
  patients.

---
## ✅ Checkpoint

1. Why does 1-NN always reach 100 % training accuracy, and why is that no reason to choose K = 1?
   <details><summary>Answer</summary>Every training sample is its own nearest neighbour (distance 0), so it is always predicted correctly (unless two identical samples have different labels). Training accuracy therefore measures memory, not generalisation; 1-NN follows the noise and its test error is usually higher than for moderate K (Section 2.2).</details>
2. With 10 three-valued loci and a binary outcome, how many free parameters do (a) the full table P(Y | X), (b) Bayes' rule without assumptions and (c) naïve Bayes need?
   <details><summary>Answer</summary>(a) $3^{10} = 59{,}049$; (b) $(3^{10}-1)\times 2 + 1 = 118{,}097$; (c) $2\cdot 10\cdot 2 + 1 = 41$.</details>
3. Using the slides' six-individual table, compute the MLE posterior $P(Y=1 \mid X = (0, 0, 1))$ (individual 1's genotype).
   <details><summary>Answer</summary>$P(Y{=}1, x) = \tfrac12\cdot\tfrac23\cdot\tfrac23\cdot\tfrac13 = \tfrac{2}{27}$; $P(Y{=}0, x) = \tfrac12\cdot 0 \cdot\tfrac13\cdot\tfrac13 = 0$ (nobody healthy has $X_1 = 0$). Posterior $= 1$ — certain, because of a zero count. With Laplace smoothing ($a=1$): $\tfrac12\cdot\tfrac36\cdot\tfrac36\cdot\tfrac26 = \tfrac1{24}$ vs $\tfrac12\cdot\tfrac16\cdot\tfrac26\cdot\tfrac26 = \tfrac1{108}$, posterior $= \tfrac{108}{108+24} \approx 0.82$.</details>
4. Explain in two sentences why the naïve-Bayes MLE requires no iterative optimisation.
   <details><summary>Answer</summary>The log of the joint likelihood is a sum of separate terms — one for $\pi$ and one per feature (and class) — that share no parameters, so each can be maximised on its own. Each piece is the log-likelihood of a coin or die (or a Gaussian), whose maximiser is a closed-form count ratio (or mean/variance).</details>
5. A colleague says: 'Naïve Bayes assumes that the genes are independent, but these two genes have correlation 0.6 across my patients, so naïve Bayes is wrong here.' Is the argument correct?
   <details><summary>Answer</summary>Not as stated. Naïve Bayes assumes independence *within each class*. Two genes can be correlated across all patients simply because both differ between ALL and AML (Section 4.1). The assumption is violated only if they remain correlated within ALL patients and within AML patients.</details>
6. What does `fit_categorical_nb(X, y, a=1.0)` return for θ[j, i, k] when class j has 4 samples, of which 0 have genotype k at locus i? And as a → ∞?
   <details><summary>Answer</summary>$(0 + 1)/(4 + 3\cdot 1) = 1/7 \approx 0.143$ instead of 0. As $a \to\infty$ every θ tends to $1/3$, the features become uninformative and the posterior tends to the prior.</details>
7. You duplicate one feature 4 times (5 identical copies in total) in a naïve-Bayes model. By how much does the log-odds change for a sample, in terms of that feature's evidence $e = \log(\theta_{i1x_i}/\theta_{i0x_i})$?
   <details><summary>Answer</summary>The evidence is added 5 times instead of once: the log-odds increases by $4e$. The sign of $e$ is unchanged, so the decision changes only if $4e$ flips the sign of the total; the probability becomes more extreme.</details>
8. How many parameters does Gaussian naïve Bayes need for the Golub data with 3 classes (say B-ALL, T-ALL, AML) and 100 genes?
   <details><summary>Answer</summary>$2nk + (k-1) = 2\cdot 100\cdot 3 + 2 = 602$.</details>

## 🧠 Questions & discussion

1. k-NN with K = 15 was close to the Bayes error on the simulated cytometry data. In which kinds of biological data (think of the number of features and of how many are irrelevant) would you expect k-NN to fail, and why?
2. The naïve-Bayes assumption is clearly false for neighbouring SNPs. Is it more or less plausible for expression levels of genes in the same pathway? For the pixels of a histology image? What could you do instead of dropping the assumption altogether?
3. Laplace smoothing puts one imaginary sample on every genotype. If you knew the population allele frequencies (e.g. from gnomAD), how could you design a more informative prior? What would its 'imaginary samples' look like?
4. Gaussian naïve Bayes failed on 50 Golub genes because of genes at the detection floor. Which other preprocessing artefacts of biological data (batch effects, saturation, dropouts) could break the Gaussian assumption, and how would you detect them?
5. A clinician wants a calibrated probability of AML, not just a label. Based on Sections 9 and 10, would you trust naïve Bayes' probabilities? What evidence would you want to see first?

## 🚀 Bonus — going further

- **Deep dive (repository):** [`../../GenerativeDiscriminativeClassification/`](../../GenerativeDiscriminativeClassification/) compares k-NN, Gaussian naïve Bayes and logistic regression on the Wisconsin breast-cancer data, with ROC curves and reliability diagrams for calibration — a natural continuation of Sections 9–10.
- **MLE and MAP refresher:** [`../../FundamentalsLearningUncertainty/`](../../FundamentalsLearningUncertainty/) develops the coin/die likelihoods and Beta/Dirichlet priors used in Sections 6 and 8 in more depth.
- **Weighted k-NN.** Replace the majority vote by votes weighted by $1/\text{distance}$ (scikit-learn: `weights='distance'`). Does it change the U-shaped curve of Section 2.2?
- **A real multinomial naïve Bayes.** Classify DNA sequences (e.g. promoter vs non-promoter) from their k-mer counts with `sklearn.naive_bayes.MultinomialNB` — the model behind early text and sequence classifiers.
- **Uncertainty you can trust.** For calibrated prediction sets on top of any classifier (including naïve Bayes), see [`../../ConformalPrediction/`](../../ConformalPrediction/).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module:            08 · Classification I — k-NN and naïve Bayes
Input:             simulated 2-marker cytometry data (3 cell types); the slides' 6-individual genotype table;
                   a simulated case/control cohort with LD partners; Golub et al. 1999 (38 train / 34 test)
Output:            k-NN decision regions and error-vs-K curve; θ tables (MLE and MAP); posteriors for all
                   27 genotypes; log-loss vs number of correlated SNPs; Gaussian-NB test errors on Golub
Method:            k-NN (majority vote, Euclidean distance); categorical naïve Bayes (MLE by counting,
                   MAP with pseudo-counts); Gaussian naïve Bayes (per-class means/variances, log space)
Key equation:      P(Y | X) ∝ P(Y) Π_i P(X_i | Y);  θ_ijk = #(X_i = k, Y = j) / #(Y = j)
Key parameters:    K = ___ ; pseudo-count a = ___ ; number of genes G = ___ ; variance floor = ___
What changed when I modified … (K / scaling / a / number of LD partners / G): ___
One failure mode:  ___ (e.g. zero counts or zero variances veto all other evidence)
Connection to 02-620 slides: 003_classification-kNN-NB slides ___
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists*, Spring 2025 lecture slides, deck `003_classification-kNN-NB` (course design S. Kim; instructors M. Xu and M. Zhang).
- Mitchell T. M. (1997) *Machine Learning*. McGraw-Hill — Chapter 6 (Bayesian learning, the naïve Bayes classifier, m-estimates) and Chapter 8 (instance-based learning, k-NN).
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — §3.5 (naïve Bayes classifiers), §1.4.2 (k-NN), §3.3–3.4 (Beta–binomial and Dirichlet–multinomial models).
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — §2.3 and §13.3 (nearest neighbours), §6.6.3 (naïve Bayes).
- James G., Witten D., Hastie T., Tibshirani R. (2013) *An Introduction to Statistical Learning*. Springer — §2.2.3 (the Bayes classifier and k-NN; source of the 3-class figures on slides 6–10).
- Golub T. R. *et al.* (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537.
- Dudoit S., Fridlyand J., Speed T. P. (2002) Comparison of discrimination methods for the classification of tumors using gene expression data. *Journal of the American Statistical Association* 97:77–87.
- Ambroise C., McLachlan G. J. (2002) Selection bias in gene extraction on the basis of microarray gene-expression data. *PNAS* 99:6562–6566.
- Domingos P., Pazzani M. (1997) On the optimality of the simple Bayesian classifier under zero-one loss. *Machine Learning* 29:103–130.

## ➡️ Next

In naïve Bayes we modelled how each class *generates* the data and then inverted the model with Bayes' rule.
Section 9 showed that its posterior is a logistic function of a sum of per-feature evidence terms. What if
we skip the generative model and learn those weights directly, by maximising $P(Y \mid X)$? That is
**logistic regression**, the discriminative classifier of the next lecture:
➡️ [Module 09 · Classification II: logistic regression](09_logistic_regression.ipynb).